# AmbSoc / RenovaBio — notebook consolidado local e multiescala — v4.2 final

**Objetivo:** produzir, em PT-BR e EN, as tabelas, figuras e diagnósticos finais do artigo histórico-territorial sobre governança, fronteira canavieira, topografia, uso anterior da terra e desempenho no RenovaBio.

Esta versão consolida os ajustes de robustez multibuffer, padronização temporal por períodos institucionais, KNN=6 para diagnósticos espaciais e figuras print-ready.


In [ ]:

# ============================================================
# 0. CONFIGURAÇÃO DE AMBIENTE, CAMINHOS E PARÂMETROS
#
# Refatorado para execução reprodutível a partir de um clone do
# repositório, sem dependência do Google Drive.
#
# Fonte única de parâmetros: `pipeline.yaml` na raiz do repositório.
# Se o arquivo não existir, aplicam-se os padrões definidos abaixo.
# ============================================================

import json
import math
import os
import re
import shutil
import sys
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt

from shapely.geometry import shape
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Image as DisplayImage

import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import OLSInfluence
from statsmodels.robust.norms import HuberT
from patsy import bs

from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill

warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 0.1 Raiz do repositório
# ------------------------------------------------------------
# Ordem de precedência:
#   1. variável de ambiente AMBSOC_REPO_ROOT
#   2. raiz detectada a partir do diretório de trabalho
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if os.environ.get('AMBSOC_REPO_ROOT'):
    ROOT = Path(os.environ['AMBSOC_REPO_ROOT']).expanduser().resolve()
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / 'requirements.txt').exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

# ------------------------------------------------------------
# 0.2 Estrutura de diretórios do repositório
# ------------------------------------------------------------
DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
OUT_DIR        = ROOT / 'outputs'

TABLES_DIR     = OUT_DIR / 'tables'
MAPS_DIR       = OUT_DIR / 'figures'
DEM_DIR        = OUT_DIR / 'dem_selection'
CORRECTED_DIR  = DEM_DIR / 'corrected_annual_selection'
NEXT_DIR       = OUT_DIR / 'next_analyses'
ORIGIN_DIR     = NEXT_DIR / 'previous_land_use'

# O pacote consolidado do artigo compartilha a árvore de outputs.
FINAL_DIR         = OUT_DIR
FINAL_TABLES      = TABLES_DIR
FINAL_FIGURES     = OUT_DIR / 'figures'
FINAL_DIAGNOSTICS = OUT_DIR / 'diagnostics'
FINAL_ROBUSTNESS  = OUT_DIR / 'robustness'

FINAL_FIGURES_PTBR = FINAL_FIGURES / 'ptbr'
FINAL_FIGURES_EN   = FINAL_FIGURES / 'en'
FINAL_TABLES_PTBR  = FINAL_TABLES / 'ptbr'
FINAL_TABLES_EN    = FINAL_TABLES / 'en'

# Mantido por compatibilidade com células que o referenciam.
CODE_DIR = DATA_PROCESSED

for p in [
    DATA_PROCESSED, TABLES_DIR, MAPS_DIR, DEM_DIR, CORRECTED_DIR,
    NEXT_DIR, ORIGIN_DIR, FINAL_DIAGNOSTICS, FINAL_ROBUSTNESS,
    FINAL_FIGURES_PTBR, FINAL_FIGURES_EN, FINAL_TABLES_PTBR, FINAL_TABLES_EN,
]:
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 0.3 Parâmetros — pipeline.yaml é a fonte canônica
# ------------------------------------------------------------
CFG_DEFAULT = {
    # Temporalidade
    'main_schema': 'pre_renovabio_1985_2015',

    # Escala principal: 25 km — única com cache exato completo para
    # fronteira, topografia e uso anterior.
    'main_buffer_km': 25,
    'legacy_buffer_km': 25,

    # Robustez multiescala: aproximada a partir da grade de 50 km do GEE.
    'buffer_candidates_km': [25, 30, 50, 100, 150],

    # Substituição do buffer PRINCIPAL por outro quando seu cache não existe.
    # Em False, a ausência interrompe a execução em vez de ser contornada
    # em silêncio. Não confundir com a aproximação dos buffers de robustez,
    # abaixo, que é o método declarado no manuscrito.
    'fallback_to_available_buffer': False,
    'build_approx_frontier_from_origin_grid': True,
    'prefer_exact_frontier_cache': True,
    'run_m3_in_buffer_robustness': True,

    # Uso anterior
    'origin_model_year_min': 1987,
    'origin_model_year_max': 2015,
    'run_unit_origin_approximation_from_grid': True,
    'n_boot_origin': 1500,
    'origin_seed': 20260609,
    'save_origin_draws': True,

    # Pesos espaciais e inferência
    'k_neighbors': 6,
    'k_neighbors_robustness': [4, 6, 8, 10],
    'permutations': 999,
    'fdr_alpha': 0.05,

    # Semente da inferência por permutação (Moran global, LISA).
    # O RNG do `esda` é global: sem semeadura os p-valores variam entre
    # execuções. Ver seção 0.5.
    'spatial_seed': 20260609,

    # Sistema de coordenadas das usinas.
    # Declarado explicitamente. A rotina de inferência permanece disponível
    # como verificação, mas não decide mais o valor usado.
    'unit_coord_crs': 'EPSG:5880',   # SIRGAS 2000 / Brazil Polyconic
    'analysis_crs': 'EPSG:5880',

    # Exportação
    'figure_dpi': 600,
    'save_svg': True,
}

CFG = dict(CFG_DEFAULT)
_yaml_path = ROOT / 'pipeline.yaml'
_yaml_loaded = False

if _yaml_path.exists():
    try:
        import yaml
        with open(_yaml_path, encoding='utf-8') as fh:
            _y = yaml.safe_load(fh) or {}
        for _bloco in ('temporalidade', 'buffers', 'uso_anterior',
                       'pesos_espaciais', 'crs', 'exportacao'):
            for _k, _v in (_y.get(_bloco) or {}).items():
                if _v != 'PREENCHER' and not (
                    isinstance(_v, str) and 'PREENCHER' in _v
                ):
                    CFG[_k] = _v
        _yaml_loaded = True
    except Exception as _exc:
        print(f'AVISO: falha ao ler pipeline.yaml ({_exc}). Usando padrões.')

CFG['buffer_km'] = CFG['main_buffer_km']   # compatibilidade

# ------------------------------------------------------------
# 0.4 CRS declarado
# ------------------------------------------------------------
# Verificado independentemente: as coordenadas X/Y das usinas, interpretadas
# em EPSG:5880, situam-se entre lon -59,5 e -43,2 e lat -23,8 e -12,3 — o
# recorte Centro-Sul. Qualquer outro candidato projetaria os pontos fora do
# território analisado.
UNIT_COORD_CRS_USED = CFG['unit_coord_crs']

# ------------------------------------------------------------
# 0.5 Semeadura da inferência por permutação
# ------------------------------------------------------------
def seed_spatial_inference(offset: int = 0) -> None:
    """Semeia o RNG global usado por `esda` em Moran e Moran_Local.

    Chamar imediatamente antes de cada estimativa por permutação. O `offset`
    permite semente distinta por teste, preservando independência entre eles
    sem abrir mão da reprodutibilidade.
    """
    np.random.seed(CFG['spatial_seed'] + offset)


np.random.seed(CFG['spatial_seed'])

# ------------------------------------------------------------
# 0.6 Gate 0 — insumos obrigatórios
# ------------------------------------------------------------
REQUIRED_INPUTS = {
    'painel_canonico_wide.csv':   'Painel de usinas: NEEA, vol%, coordenadas X/Y (EPSG:5880), CNPJ',
    'crosswalk_centrosul.csv':    'Crosswalk usina -> município sede',
    'mill_confounder_base.csv':   'Confundidores em nível de usina',
    'base_psm_integrada_raw.csv': 'Base municipal de indicadores ex-ante',
    'centro_sul_munis.geojson':   'Geometrias municipais do recorte Centro-Sul',
}

_missing = [f for f in REQUIRED_INPUTS if not (DATA_PROCESSED / f).exists()]

# ------------------------------------------------------------
# 0.7 Rótulos e ordenações
# ------------------------------------------------------------
LANGS = ['ptbr', 'en']

PERIOD_ORDER = [
    'pre_law_1987_2001',
    'post_law_pre_zae_2002_2008',
    'zae_2009_2019',
    'post_zae_2020_2023',
]

PERIOD_LABELS_PTBR = {
    'pre_law_1987_2001': 'Pré-Lei, 1987–2001',
    'post_law_pre_zae_2002_2008': 'Pós-Lei / pré-ZAE, 2002–2008',
    'zae_2009_2019': 'Vigência do ZAE-Cana, 2009–2019',
    'post_zae_2020_2023': 'Pós-revogação, 2020–2023',
}

PERIOD_LABELS_EN = {
    'pre_law_1987_2001': 'Pre-law period, 1987–2001',
    'post_law_pre_zae_2002_2008': 'Post-law / pre-ZAE, 2002–2008',
    'zae_2009_2019': 'ZAE-Cana period, 2009–2019',
    'post_zae_2020_2023': 'Post-revocation, 2020–2023',
}

PERIOD_LABELS = PERIOD_LABELS_PTBR

PERIOD_YEARS = {
    'pre_law_1987_2001': 15,
    'post_law_pre_zae_2002_2008': 7,
    'zae_2009_2019': 11,
    'post_zae_2020_2023': 4,
}

SLOPE_ORDER = ['00_03', '03_06', '06_09', '09_12', '12_20', 'gt_20']

SLOPE_LABELS_PTBR = {
    '00_03': '0–3%',
    '03_06': '>3–6%',
    '06_09': '>6–9%',
    '09_12': '>9–12%',
    '12_20': '>12–20%',
    'gt_20': '>20%',
}

SLOPE_LABELS_EN = dict(SLOPE_LABELS_PTBR)
SLOPE_LABELS = SLOPE_LABELS_PTBR

ORIGIN_LABELS_PTBR = {
    1: 'Pastagem',
    2: 'Agricultura anual',
    3: 'Agricultura perene',
    4: 'Mosaico agropecuário',
    5: 'Silvicultura',
    6: 'Vegetação nativa',
    7: 'Outros usos',
}

ORIGIN_LABELS_EN = {
    1: 'Pasture',
    2: 'Annual agriculture',
    3: 'Perennial agriculture',
    4: 'Agricultural mosaic',
    5: 'Forestry',
    6: 'Native vegetation',
    7: 'Other uses',
}

ORIGIN_LABELS = ORIGIN_LABELS_PTBR

FRONTIER_PERIOD_LABELS_PTBR = {
    'frontier_until_2001': 'Até 2001',
    'frontier_2002_2008': '2002–2008',
    'frontier_2009_2015': '2009–2015',
    'frontier_after_2015': 'Depois de 2015',
}

FRONTIER_PERIOD_LABELS_EN = {
    'frontier_until_2001': 'Until 2001',
    'frontier_2002_2008': '2002–2008',
    'frontier_2009_2015': '2009–2015',
    'frontier_after_2015': 'After 2015',
}

# ------------------------------------------------------------
# 0.8 Relatório de ambiente
# ------------------------------------------------------------
print('=' * 70)
print('AMBIENTE')
print('=' * 70)
print(f'  Colab .................. {IN_COLAB}')
print(f'  Raiz do repositório .... {ROOT}')
print(f'  Parâmetros ............. {"pipeline.yaml" if _yaml_loaded else "padrões internos"}')
print(f'  Python ................. {sys.version.split()[0]}')
print(f'  geopandas / esda ....... {gpd.__version__} / {__import__("esda").__version__}')

print('\nPARÂMETROS CRÍTICOS')
print(f'  CRS das usinas ......... {CFG["unit_coord_crs"]}  (declarado)')
print(f'  Buffer principal ....... {CFG["main_buffer_km"]} km')
print(f'  Fallback de buffers .... {CFG["fallback_to_available_buffer"]}')
print(f'  Semente — bootstrap .... {CFG["origin_seed"]}')
print(f'  Semente — permutação ... {CFG["spatial_seed"]}')
print(f'  Permutações ............ {CFG["permutations"]}')
print(f'  KNN principal .......... {CFG["k_neighbors"]}')

print('\nINSUMOS OBRIGATÓRIOS')
for _f, _desc in REQUIRED_INPUTS.items():
    _ok = 'OK  ' if (DATA_PROCESSED / _f).exists() else 'FALTA'
    print(f'  [{_ok}] {_f:30s} {_desc[:40]}')

if _missing:
    raise FileNotFoundError(
        '\n\nGATE 0 — insumos obrigatórios ausentes em data/processed/:\n  '
        + '\n  '.join(_missing)
        + '\n\nO pipeline para aqui por desenho: nenhum insumo canônico é '
          'substituído ou simulado. Ver data/raw/MANIFEST.md.\n'
    )

if CFG['fallback_to_available_buffer']:
    print('\nAVISO: fallback de buffers ATIVO. Caches ausentes serão '
          'substituídos por aproximação a partir da grade de origem, '
          'tornando os resultados dependentes do estado do disco. '
          'Para o pacote publicado, use False.')

print('\nGate 0: aprovado. Ambiente pronto.')
print('=' * 70)



## 1. Funções auxiliares e auditoria dos caches

A auditoria localiza os arquivos por caminhos canônicos e, quando necessário, por busca recursiva. Arquivos ausentes em módulos opcionais geram avisos, sem interromper as demais análises.


In [ ]:

# ============================================================
# 1.1. FUNÇÕES AUXILIARES
# ============================================================
def normalize_cnpj(series):
    return (
        series.astype('string')
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .str.replace(r'\D', '', regex=True)
        .str.zfill(14)
    )


def first_existing(paths, required=True, label='arquivo'):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    if required:
        raise FileNotFoundError(
            f'Não encontrei {label}. Caminhos testados:\n'
            + '\n'.join(str(Path(p)) for p in paths)
        )
    return None


def first_rglob(root, pattern, required=True, label='arquivo'):
    matches = sorted(Path(root).rglob(pattern))
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(
            f'Não encontrei {label} com padrão {pattern} em {root}'
        )
    return None


def read_csv(path, **kwargs):
    path = Path(path)
    print('Lendo:', path)
    return pd.read_csv(path, low_memory=False, **kwargs)


def set_publication_style():
    mpl.rcParams.update({
        'figure.dpi': 120,
        'savefig.dpi': CFG['figure_dpi'],
        'font.size': 10,
        'axes.titlesize': 12,
        'axes.labelsize': 10,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 9,
        'legend.title_fontsize': 9,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.grid': False,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'svg.fonttype': 'none',
    })


set_publication_style()


def save_figure(fig, stem, dpi=None):
    # Salva figura genérica no diretório raiz de figuras.
    dpi = dpi or CFG['figure_dpi']
    png = FINAL_FIGURES / f'{stem}.png'
    pdf = FINAL_FIGURES / f'{stem}.pdf'
    fig.savefig(png, dpi=dpi, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    if CFG.get('save_svg', True):
        svg = FINAL_FIGURES / f'{stem}.svg'
        fig.savefig(svg, bbox_inches='tight', facecolor='white')
        print('Salvo:', png.name, pdf.name, svg.name)
    else:
        print('Salvo:', png.name, 'e', pdf.name)
    return png, pdf


def save_lang_figure(fig, stem, lang='ptbr', dpi=None):
    # Salva versões print-ready por idioma: PNG 600 dpi, PDF e SVG.
    dpi = dpi or CFG['figure_dpi']
    out_dir = FINAL_FIGURES_PTBR if lang == 'ptbr' else FINAL_FIGURES_EN
    out_dir.mkdir(parents=True, exist_ok=True)

    png = out_dir / f'{stem}_{lang}.png'
    pdf = out_dir / f'{stem}_{lang}.pdf'
    fig.savefig(png, dpi=dpi, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    paths = [png, pdf]

    if CFG.get('save_svg', True):
        svg = out_dir / f'{stem}_{lang}.svg'
        fig.savefig(svg, bbox_inches='tight', facecolor='white')
        paths.append(svg)

    print('Salvo:', ', '.join(p.name for p in paths))
    return paths


def tidy_model(model, outcome, model_name):
    ci = model.conf_int()
    return pd.DataFrame({
        'outcome': outcome,
        'model': model_name,
        'term': model.params.index,
        'coef': model.params.values,
        'std_err_hc3': model.bse.values,
        'p_value': model.pvalues.values,
        'ci_low': ci.iloc[:, 0].values,
        'ci_high': ci.iloc[:, 1].values,
        'n': int(model.nobs),
        'r2': getattr(model, 'rsquared', np.nan),
        'r2_adj': getattr(model, 'rsquared_adj', np.nan),
        'aic': getattr(model, 'aic', np.nan),
        'bic': getattr(model, 'bic', np.nan),
    })


def print_status(label, path):
    mark = '✓' if path is not None and Path(path).exists() else '—'
    print(f'{mark} {label}: {path}')


def model_data_candidates(buffer_km, include_generic=False):
    paths = [
        TABLES_DIR / f'overlay_frontier_merged_long_{buffer_km}km.csv',
        TABLES_DIR / f'frontier_analysis_long_{buffer_km}km.csv',
        OUT_DIR / f'overlay_frontier_merged_long_{buffer_km}km.csv',
        OUT_DIR / f'frontier_analysis_long_{buffer_km}km.csv',
    ]
    if include_generic:
        paths.extend([
            TABLES_DIR / 'overlay_frontier_merged_long.csv',
            TABLES_DIR / 'frontier_analysis_long.csv',
        ])
    return paths


def frontier_candidates(buffer_km, include_generic=False):
    paths = [
        TABLES_DIR / f'frontier_temporal_robustness_all_schemas_{buffer_km}km.csv',
        OUT_DIR / f'frontier_temporal_robustness_all_schemas_{buffer_km}km.csv',
    ]
    if include_generic:
        paths.extend([
            TABLES_DIR / 'frontier_temporal_robustness_all_schemas.csv',
            OUT_DIR / 'frontier_temporal_robustness_all_schemas.csv',
        ])
    return paths


def dem_candidates(buffer_km, include_generic=False):
    paths = [
        DEM_DIR / f'dem_frontier_all_schemas_{buffer_km}km.csv',
        OUT_DIR / f'dem_frontier_all_schemas_{buffer_km}km.csv',
        DEM_DIR / f'dem_frontier_{buffer_km}km.csv',
        OUT_DIR / f'dem_frontier_{buffer_km}km.csv',
    ]
    if include_generic:
        paths.extend([
            DEM_DIR / 'dem_frontier_all_schemas.csv',
        ])
    return paths


def get_labels(lang):
    if lang == 'en':
        return {
            'period': PERIOD_LABELS_EN,
            'slope': SLOPE_LABELS_EN,
            'origin': ORIGIN_LABELS_EN,
            'frontier_period': FRONTIER_PERIOD_LABELS_EN,
        }
    return {
        'period': PERIOD_LABELS_PTBR,
        'slope': SLOPE_LABELS_PTBR,
        'origin': ORIGIN_LABELS_PTBR,
        'frontier_period': FRONTIER_PERIOD_LABELS_PTBR,
    }


def add_panel_grid(ax, axis='y', alpha=.20):
    ax.grid(axis=axis, alpha=alpha, linewidth=.6)
    ax.set_axisbelow(True)


def safe_to_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print('Tabela salva:', path)


def write_bilingual_table(df, stem, pt_labels=None, en_labels=None):
    # Exporta uma tabela base e versões com nomes de colunas em PT-BR e EN.
    safe_to_csv(df, FINAL_TABLES / f'{stem}.csv')
    if pt_labels:
        safe_to_csv(df.rename(columns=pt_labels), FINAL_TABLES_PTBR / f'{stem}_ptbr.csv')
    if en_labels:
        safe_to_csv(df.rename(columns=en_labels), FINAL_TABLES_EN / f'{stem}_en.csv')


In [ ]:

# ============================================================
# 1.2. INVENTÁRIO DOS ARQUIVOS LOCAIS
# ============================================================
panel_path = first_existing([
    CODE_DIR / 'painel_canonico_wide_corrigido.csv',
    CODE_DIR / 'painel_canonico_wide.csv',
], required=True, label='painel canônico')

model_data_paths_by_buffer = {}
frontier_paths_by_buffer = {}
dem_paths_by_buffer = {}

for b in CFG['buffer_candidates_km']:
    model_data_paths_by_buffer[b] = first_existing(
        model_data_candidates(b, include_generic=False),
        required=False
    )
    frontier_paths_by_buffer[b] = first_existing(
        frontier_candidates(b, include_generic=False),
        required=False
    )
    dem_paths_by_buffer[b] = first_existing(
        dem_candidates(b, include_generic=False),
        required=False
    )

# Caches genéricos: normalmente correspondem à versão legada do notebook.
generic_model_data_path = first_existing(
    model_data_candidates(CFG['legacy_buffer_km'], include_generic=True),
    required=False
)
generic_frontier_path = first_existing(
    frontier_candidates(CFG['legacy_buffer_km'], include_generic=True),
    required=False
)
generic_dem_path = first_existing(
    dem_candidates(CFG['legacy_buffer_km'], include_generic=True),
    required=False
)

main_b = CFG['main_buffer_km']
legacy_b = CFG['legacy_buffer_km']

if model_data_paths_by_buffer.get(main_b) is not None or frontier_paths_by_buffer.get(main_b) is not None:
    active_buffer_km = main_b
elif CFG['fallback_to_available_buffer'] and (
    model_data_paths_by_buffer.get(legacy_b) is not None
    or frontier_paths_by_buffer.get(legacy_b) is not None
    or generic_model_data_path is not None
    or generic_frontier_path is not None
):
    active_buffer_km = legacy_b
    print(
        f'AVISO: cache principal de {main_b} km não encontrado. '
        f'Usando {legacy_b} km como fallback.'
    )
else:
    active_buffer_km = main_b

CFG['active_buffer_km'] = active_buffer_km
CFG['buffer_km'] = active_buffer_km

model_data_path = (
    model_data_paths_by_buffer.get(active_buffer_km)
    or (generic_model_data_path if active_buffer_km == legacy_b else None)
)

frontier_path = (
    frontier_paths_by_buffer.get(active_buffer_km)
    or (generic_frontier_path if active_buffer_km == legacy_b else None)
)

dem_path = (
    dem_paths_by_buffer.get(active_buffer_km)
    or generic_dem_path
)

map_png = first_existing([
    MAPS_DIR / 'mapa_fronteira_cana_ptbr.png',
    MAPS_DIR / 'mapa_fronteira_cana.png',
], required=False)

map_pdf = first_existing([
    MAPS_DIR / 'mapa_fronteira_cana_ptbr.pdf',
    MAPS_DIR / 'mapa_fronteira_cana.pdf',
], required=False)

map_png_en = first_existing([
    MAPS_DIR / 'mapa_fronteira_cana_en.png',
    MAPS_DIR / 'mapa_fronteira_cana_english.png',
], required=False)

map_pdf_en = first_existing([
    MAPS_DIR / 'mapa_fronteira_cana_en.pdf',
    MAPS_DIR / 'mapa_fronteira_cana_english.pdf',
], required=False)

expected_topography = {
    'annual': CORRECTED_DIR / 'annual_topographic_selection_with_bootstrap.csv',
    'period': CORRECTED_DIR / 'period_topographic_selection_summary_corrected.csv',
    'contrasts': CORRECTED_DIR / 'selection_ratio_period_contrasts_bootstrap.csv',
    'segmented_coef': CORRECTED_DIR / 'annual_segmented_regression_coefficients.csv',
    'segmented_slopes': CORRECTED_DIR / 'annual_segmented_regression_implied_slopes.csv',
    'entry_origin': CORRECTED_DIR / 'entry_origin_coverage_corrected.csv',
    'initial_stock': CORRECTED_DIR / 'initial_stock_slope_distribution.csv',
}

origin_export_candidates = [
    p for p in ORIGIN_DIR.rglob('previous_land_use_*.csv')
    if re.fullmatch(r'previous_land_use_\d{4}_\d{4}\.csv', p.name)
]

print_status('Painel canônico', panel_path)
print_status(f'Base analítica longa — buffer ativo {active_buffer_km} km', model_data_path)
print_status(f'Fronteira — buffer ativo {active_buffer_km} km', frontier_path)
print_status(f'DEM/topografia — buffer ativo {active_buffer_km} km', dem_path)
print_status('Mapa de fronteira PT-BR', map_png)
print_status('Mapa de fronteira EN', map_png_en)

print('\nDisponibilidade de caches multiescala:')
for b in CFG['buffer_candidates_km']:
    print(
        f'  {b:>3} km | '
        f'base={model_data_paths_by_buffer.get(b) is not None} | '
        f'fronteira={frontier_paths_by_buffer.get(b) is not None} | '
        f'DEM={dem_paths_by_buffer.get(b) is not None}'
    )

print('\nCaches da seleção topográfica por período institucional:')
for name, path in expected_topography.items():
    print_status(name, path)

print(
    f'\nArquivos brutos de uso anterior encontrados: '
    f'{len(origin_export_candidates)}/8'
)
for p in origin_export_candidates:
    print('  ', p)



## 2. Base analítica canônica

A base preferencial é `overlay_frontier_merged_long.csv`, que contém os dois outcomes e os três esquemas temporais. Caso ela não exista, o notebook a reconstrói localmente a partir do painel canônico e das estatísticas de fronteira já extraídas.


In [ ]:

# ============================================================
# 2.1. CARREGAR OU RECONSTRUIR A BASE LONGA
# ============================================================
panel = read_csv(panel_path, dtype={'cnpj_clean': 'string'})
panel['cnpj_clean'] = normalize_cnpj(panel['cnpj_clean'])

if model_data_path is not None:
    analysis_all = read_csv(
        model_data_path,
        dtype={'cnpj_clean': 'string'}
    )
    analysis_all['cnpj_clean'] = normalize_cnpj(
        analysis_all['cnpj_clean']
    )

    if 'buffer_km' not in analysis_all.columns:
        analysis_all['buffer_km'] = CFG['active_buffer_km']
else:
    if frontier_path is None:
        raise FileNotFoundError(
            'A base longa não existe e também não encontrei '
            'as estatísticas locais da fronteira para o buffer ativo.'
        )

    frontier_all = read_csv(
        frontier_path,
        dtype={'cnpj_clean': 'string'}
    )
    frontier_all['cnpj_clean'] = normalize_cnpj(
        frontier_all['cnpj_clean']
    )

    if 'buffer_km' not in frontier_all.columns:
        frontier_all['buffer_km'] = CFG['active_buffer_km']

    metadata = [
        c for c in [
            'cnpj_clean', 'emissor', 'cidade', 'uf', 'rota',
            'rota_tag', 'cohort', 'X', 'Y', 'longitude', 'latitude',
            'neea_first', 'neea_t2_2026', 'delta_neea',
            'vol_pct_first', 'vol_pct_t2_2026', 'delta_vol_pct'
        ]
        if c in panel.columns
    ]

    base = panel[metadata].drop_duplicates('cnpj_clean').copy()

    merged = base.merge(
        frontier_all,
        on='cnpj_clean',
        how='inner',
        validate='one_to_many',
        suffixes=('', '_frontier')
    )

    neea = merged.copy()
    neea['outcome'] = 'delta_neea'

    vol = merged.copy()
    vol['outcome'] = 'delta_vol_pct'

    analysis_all = pd.concat([neea, vol], ignore_index=True)

# Eliminar colunas topográficas incorporadas por um esquema incorreto.
for c in [
    'share_cane_slope_le12',
    'mean_cane_slope_pct',
    'mean_cane_elevation_m',
]:
    if c in analysis_all.columns:
        analysis_all = analysis_all.drop(columns=c)

# Incorporar a topografia correta por CNPJ × esquema.
if dem_path is not None:
    dem_all = read_csv(
        dem_path,
        dtype={'cnpj_clean': 'string'}
    )
    dem_all['cnpj_clean'] = normalize_cnpj(dem_all['cnpj_clean'])

    if 'buffer_km' not in dem_all.columns:
        dem_all['buffer_km'] = CFG['active_buffer_km']

    dem_keep = [
        c for c in [
            'cnpj_clean', 'schema', 'buffer_km',
            'share_cane_slope_le12',
            'mean_cane_slope_pct',
            'mean_cane_elevation_m'
        ]
        if c in dem_all.columns
    ]

    dem_all = (
        dem_all[dem_keep]
        .drop_duplicates(['cnpj_clean', 'schema', 'buffer_km'])
    )

    merge_keys = ['cnpj_clean', 'schema']
    if 'buffer_km' in analysis_all.columns and 'buffer_km' in dem_all.columns:
        merge_keys.append('buffer_km')

    analysis_all = analysis_all.merge(
        dem_all,
        on=merge_keys,
        how='left',
        validate='many_to_one'
    )
else:
    print('AVISO: DEM/topografia não encontrado; modelos sem topografia.')

print('Base consolidada:', analysis_all.shape)
print('Unidades:', analysis_all['cnpj_clean'].nunique())
print('Buffer ativo:', CFG['active_buffer_km'], 'km')
print('Esquemas:', analysis_all['schema'].dropna().unique())
print('Outcomes:', analysis_all['outcome'].dropna().unique())


In [ ]:
# ============================================================
# 2.2. PADRONIZAR COLUNAS E CRIAR AMOSTRAS PRINCIPAIS
# ============================================================
uf_col = 'uf_x' if 'uf_x' in analysis_all.columns else (
    'uf' if 'uf' in analysis_all.columns else 'uf_y'
)

# Preferir longitude/latitude explícitas; usar X/Y apenas como fallback.
if {'longitude', 'latitude'}.issubset(analysis_all.columns):
    x_col, y_col = 'longitude', 'latitude'
elif {'X', 'Y'}.issubset(analysis_all.columns):
    x_col, y_col = 'X', 'Y'
else:
    raise ValueError('Não encontrei colunas de coordenadas: longitude/latitude ou X/Y.')

numeric_cols = [
    'delta_neea', 'delta_vol_pct',
    'neea_first', 'vol_pct_first',
    'mean_first_key_year', 'cane_area_total_ha',
    'share_cane_slope_le12',
    'buffer_km',
    x_col, y_col
]

for col in numeric_cols:
    if col in analysis_all.columns:
        analysis_all[col] = pd.to_numeric(
            analysis_all[col],
            errors='coerce'
        )

analysis_all['log_cane_area'] = np.log1p(
    analysis_all['cane_area_total_ha'].clip(lower=0)
)

main = analysis_all.loc[
    analysis_all['schema'].astype(str).eq(CFG['main_schema'])
].copy()

# Uma linha por CNPJ em cada outcome.
main = (
    main.sort_values(['outcome', 'cnpj_clean'])
    .drop_duplicates(['outcome', 'cnpj_clean'])
)

main['frontier_year_c'] = (
    main['mean_first_key_year']
    - main['mean_first_key_year'].mean()
)

# Categoria institucional, usada apenas para descrição/figuras suplementares.
frontier_bins = [-np.inf, 2001, 2008, 2015, np.inf]
frontier_codes = [
    'frontier_until_2001',
    'frontier_2002_2008',
    'frontier_2009_2015',
    'frontier_after_2015',
]

main['frontier_period_generation'] = pd.cut(
    main['mean_first_key_year'],
    bins=frontier_bins,
    labels=frontier_codes,
    right=True
)

# Categoria legada mantida apenas como diagnóstico, não como eixo narrativo.
main['frontier_generation_legacy'] = pd.cut(
    main['mean_first_key_year'],
    bins=[-np.inf, 1995, 2005, np.inf],
    labels=['Até 1995', '1996–2005', '2006–2015'],
    right=True
)

print('Amostra principal:', main.shape)
print('Coordenadas usadas:', x_col, y_col)
display(
    main.groupby('outcome').agg(
        n=('cnpj_clean', 'nunique'),
        frontier_valid=('mean_first_key_year', 'count'),
        topography_valid=('share_cane_slope_le12', 'count'),
    )
)

# Estatísticas descritivas da amostra analítica, separadas por outcome.
# Isso evita que a tabela T1 duplique as unidades por causa da base longa.
descriptive_specs = {
    'delta_neea': [
        'delta_neea', 'neea_first', 'mean_first_key_year',
        'cane_area_total_ha', 'share_cane_slope_le12'
    ],
    'delta_vol_pct': [
        'delta_vol_pct', 'vol_pct_first', 'mean_first_key_year',
        'cane_area_total_ha', 'share_cane_slope_le12'
    ],
}

desc_rows = []
for outcome_name, cols in descriptive_specs.items():
    sample = main.loc[main['outcome'].astype(str).eq(outcome_name)].copy()
    for var in [c for c in cols if c in sample.columns]:
        s = pd.to_numeric(sample[var], errors='coerce').dropna()
        desc_rows.append({
            'analytic_sample': outcome_name,
            'variable': var,
            'n': int(s.shape[0]),
            'mean': s.mean(),
            'std': s.std(ddof=1),
            'min': s.min(),
            'p25': s.quantile(.25),
            'median': s.median(),
            'p75': s.quantile(.75),
            'max': s.max(),
        })

descriptives = pd.DataFrame(desc_rows)

descriptives.to_csv(
    FINAL_TABLES / 'Tabela_1_estatisticas_descritivas_amostra_analitica.csv',
    index=False,
    encoding='utf-8-sig'
)

display(descriptives.round(4))



## 3. Geografia histórica da fronteira

O mapa de publicação já produzido é reutilizado diretamente do cache. A área aproximada nos anos-chave também é recuperada do resultado anterior, evitando nova consulta ao MapBiomas.


In [ ]:

# ============================================================
# 3.1. MAPA DE FRONTEIRA + UNIDADES CERTIFICADAS
# ============================================================
if map_png is not None:
    target_png = FINAL_FIGURES_PTBR / 'Figura_1_mapa_fronteira_e_usinas_ptbr.png'
    shutil.copy2(map_png, target_png)

    # Cópia também no diretório raiz para compatibilidade.
    shutil.copy2(map_png, FINAL_FIGURES / 'Figura_1_mapa_fronteira_e_usinas.png')

    if map_pdf is not None:
        shutil.copy2(
            map_pdf,
            FINAL_FIGURES_PTBR / 'Figura_1_mapa_fronteira_e_usinas_ptbr.pdf'
        )
        shutil.copy2(
            map_pdf,
            FINAL_FIGURES / 'Figura_1_mapa_fronteira_e_usinas.pdf'
        )

    display(DisplayImage(filename=str(target_png)))
else:
    print(
        'AVISO: mapa PT-BR de publicação não localizado. '
        'Os demais módulos continuarão normalmente.'
    )

if map_png_en is not None:
    target_png_en = FINAL_FIGURES_EN / 'Figure_1_frontier_map_and_certified_units_en.png'
    shutil.copy2(map_png_en, target_png_en)

    if map_pdf_en is not None:
        shutil.copy2(
            map_pdf_en,
            FINAL_FIGURES_EN / 'Figure_1_frontier_map_and_certified_units_en.pdf'
        )

    display(DisplayImage(filename=str(target_png_en)))
else:
    print(
        'AVISO: mapa EN não localizado. '
        'Se necessário, exporte uma versão em inglês no notebook cartográfico.'
    )


In [ ]:
# ============================================================
# 3.2. ÁREA APROXIMADA NOS ANOS-CHAVE
# ============================================================
area_cache_candidates = [
    TABLES_DIR / 'area_cana_anos_chave.csv',
    MAPS_DIR / 'area_cana_anos_chave.csv',
]

area_cache = first_existing(
    area_cache_candidates,
    required=False
)

if area_cache is not None:
    areas_key = read_csv(area_cache)
else:
    # Valores recuperados do notebook original já executado.
    # Foram calculados sobre o MapBiomas em escala diagnóstica.
    areas_key = pd.DataFrame({
        'year': [1985, 1995, 2005, 2015, 2024],
        'area_ha_approx': [
            1.514409e6,
            3.143302e6,
            5.219596e6,
            1.1635773e7,
            1.1451883e7,
        ]
    })

areas_key['area_mha_approx'] = (
    areas_key['area_ha_approx'] / 1e6
)

areas_key.to_csv(
    FINAL_TABLES / 'area_cana_anos_chave.csv',
    index=False,
    encoding='utf-8-sig'
)

for lang in LANGS:
    fig, ax = plt.subplots(figsize=(8.2, 4.6))
    ax.plot(
        areas_key['year'],
        areas_key['area_mha_approx'],
        marker='o',
        linewidth=1.8
    )

    if lang == 'ptbr':
        ax.set_xlabel('Ano-chave')
        ax.set_ylabel('Área aproximada (milhões de ha)')
        ax.set_title('Área canavieira estimada nos anos-chave, 1985–2024')
        stem = 'Figura_2_area_cana_anos_chave'
    else:
        ax.set_xlabel('Key year')
        ax.set_ylabel('Approximate area (million ha)')
        ax.set_title('Estimated sugarcane area in key years, 1985–2024')
        stem = 'Figure_2_sugarcane_area_key_years'

    ax.set_xticks(areas_key['year'])
    add_panel_grid(ax)
    fig.tight_layout()
    save_lang_figure(fig, stem, lang)
    plt.show()

display(areas_key)



## 4. Seleção topográfica da expansão

Esta seção usa diretamente os CSVs corrigidos já salvos após a extração anual de 1987–2023. Nenhum cálculo espacial é repetido.


In [ ]:

# ============================================================
# 4.1. CARREGAR RESULTADOS TOPOGRÁFICOS CORRIGIDOS
# ============================================================
missing_topography = [
    name for name, path in expected_topography.items()
    if not Path(path).exists()
]

if missing_topography:
    raise FileNotFoundError(
        'Faltam caches topográficos corrigidos: '
        + ', '.join(missing_topography)
    )

annual_topo = read_csv(expected_topography['annual'])
period_topo = read_csv(expected_topography['period'])
topo_contrasts = read_csv(expected_topography['contrasts'])
segmented_coef = read_csv(expected_topography['segmented_coef'])
segmented_slopes = read_csv(expected_topography['segmented_slopes'])
entry_origin = read_csv(expected_topography['entry_origin'])
initial_stock = read_csv(expected_topography['initial_stock'])

print('Anual:', annual_topo.shape)
print('Períodos:', period_topo.shape)
print('Contrastes:', topo_contrasts.shape)


In [ ]:
# ============================================================
# 4.2. HEATMAP DE SELEÇÃO TOPOGRÁFICA
# ============================================================
heatmap = (
    period_topo
    .pivot(index='period', columns='slope_bin', values='selection_ratio')
    .reindex(index=PERIOD_ORDER, columns=SLOPE_ORDER)
)

log2_heatmap = np.log2(heatmap)
max_abs = np.nanmax(np.abs(log2_heatmap.to_numpy()))

for lang in LANGS:
    labels = get_labels(lang)

    fig, ax = plt.subplots(figsize=(10.8, 5.8))
    image = ax.imshow(
        log2_heatmap,
        aspect='auto',
        cmap='RdBu_r',
        vmin=-max_abs,
        vmax=max_abs
    )

    ax.set_xticks(range(len(SLOPE_ORDER)))
    ax.set_xticklabels([labels['slope'][s] for s in SLOPE_ORDER])

    ax.set_yticks(range(len(PERIOD_ORDER)))
    ax.set_yticklabels([labels['period'][p] for p in PERIOD_ORDER])

    if lang == 'ptbr':
        ax.set_xlabel('Faixa de declividade')
        ax.set_ylabel('Período de primeira entrada persistente')
        ax.set_title(
            'Seleção topográfica da expansão canavieira\n'
            'Razão entre nova cana e disponibilidade de terras produtivas'
        )
        cbar_label = (
            'log₂ da razão de seleção\n'
            '(0 = proporcional à disponibilidade)'
        )
        stem = 'Figura_3_selecao_topografica_heatmap'
    else:
        ax.set_xlabel('Slope class')
        ax.set_ylabel('Period of first persistent entry')
        ax.set_title(
            'Topographic selection of sugarcane expansion\n'
            'Ratio between new sugarcane and productive-land availability'
        )
        cbar_label = (
            'log₂ selection ratio\n'
            '(0 = proportional to availability)'
        )
        stem = 'Figure_3_topographic_selection_heatmap'

    for i, period in enumerate(PERIOD_ORDER):
        for j, slope in enumerate(SLOPE_ORDER):
            value = heatmap.loc[period, slope]
            ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=9)

    cbar = fig.colorbar(image, ax=ax, fraction=.035, pad=.03)
    cbar.set_label(cbar_label)

    fig.tight_layout()
    save_lang_figure(fig, stem, lang)
    plt.show()

# Cópia de compatibilidade no diretório raiz, em PT-BR.
fig, ax = plt.subplots(figsize=(10.8, 5.8))
image = ax.imshow(log2_heatmap, aspect='auto', cmap='RdBu_r', vmin=-max_abs, vmax=max_abs)
ax.set_xticks(range(len(SLOPE_ORDER)))
ax.set_xticklabels([SLOPE_LABELS_PTBR[s] for s in SLOPE_ORDER])
ax.set_yticks(range(len(PERIOD_ORDER)))
ax.set_yticklabels([PERIOD_LABELS_PTBR[p] for p in PERIOD_ORDER])
ax.set_xlabel('Faixa de declividade')
ax.set_ylabel('Período de primeira entrada persistente')
ax.set_title('Seleção topográfica da expansão canavieira')
for i, period in enumerate(PERIOD_ORDER):
    for j, slope in enumerate(SLOPE_ORDER):
        ax.text(j, i, f'{heatmap.loc[period, slope]:.2f}', ha='center', va='center', fontsize=9)
cbar = fig.colorbar(image, ax=ax, fraction=.035, pad=.03)
cbar.set_label('log₂ da razão de seleção\n(0 = proporcional à disponibilidade)')
fig.tight_layout()
save_figure(fig, 'Figura_3_selecao_topografica_heatmap')
plt.close(fig)


In [ ]:
# ============================================================
# 4.3. EVOLUÇÃO ANUAL DA SELEÇÃO TOPOGRÁFICA
# ============================================================
# Figura principal: faixas operacionais centrais. As faixas >12% são mantidas
# em figura suplementar para não comprimir a escala visual do corpo principal.
plot_bins_main = ['03_06', '06_09', '09_12']
plot_bins_supp = ['03_06', '06_09', '09_12', '12_20', 'gt_20']

for lang in LANGS:
    labels = get_labels(lang)
    fig, ax = plt.subplots(figsize=(11, 6.0))

    for slope in plot_bins_main:
        g = (
            annual_topo.loc[annual_topo['slope_bin'].eq(slope)]
            .sort_values('year')
        )

        line = ax.plot(
            g['year'],
            np.log2(g['selection_ratio']),
            marker='o',
            markersize=3,
            linewidth=1.6,
            label=labels['slope'][slope]
        )[0]

        valid = (
            g['selection_ratio_ci_low'].gt(0)
            & g['selection_ratio_ci_high'].gt(0)
        )

        ax.fill_between(
            g.loc[valid, 'year'],
            np.log2(g.loc[valid, 'selection_ratio_ci_low']),
            np.log2(g.loc[valid, 'selection_ratio_ci_high']),
            alpha=.10,
            color=line.get_color()
        )

    if lang == 'ptbr':
        proportional = 'Uso proporcional à disponibilidade'
        events = [
            (2002, 'Lei 11.241/2002'),
            (2009, 'ZAE-Cana'),
            (2020, 'Pós-revogação'),
        ]
        ax.set_xlabel('Ano da primeira entrada persistente da cana')
        ax.set_ylabel('log₂ da razão de seleção')
        ax.set_title('Evolução anual da seleção topográfica da expansão canavieira')
        stem = 'Figura_4_evolucao_anual_selecao_topografica'
    else:
        proportional = 'Proportional to availability'
        events = [
            (2002, 'Law 11,241/2002'),
            (2009, 'ZAE-Cana'),
            (2020, 'Post-revocation'),
        ]
        ax.set_xlabel('Year of first persistent sugarcane entry')
        ax.set_ylabel('log₂ selection ratio')
        ax.set_title('Annual evolution of topographic selection in sugarcane expansion')
        stem = 'Figure_4_annual_topographic_selection'

    ax.axhline(0, linestyle='--', linewidth=1, label=proportional)

    ymin, ymax = ax.get_ylim()
    for year, label in events:
        ax.axvline(year, linestyle=':', linewidth=1)
        ax.text(year + .2, ymax, label, rotation=90, va='top', fontsize=8)
    ax.set_ylim(ymin, ymax)

    ax.legend(frameon=False, ncol=2, loc='lower left')
    add_panel_grid(ax)
    fig.tight_layout()
    save_lang_figure(fig, stem, lang)
    plt.show()

    # Figura suplementar com todas as faixas >3%.
    fig, ax = plt.subplots(figsize=(11, 6.2))
    for slope in plot_bins_supp:
        g = annual_topo.loc[annual_topo['slope_bin'].eq(slope)].sort_values('year')
        ax.plot(
            g['year'],
            np.log2(g['selection_ratio']),
            marker='o',
            markersize=2.8,
            linewidth=1.2,
            label=labels['slope'][slope]
        )

    ax.axhline(0, linestyle='--', linewidth=1, label=proportional)
    ymin, ymax = ax.get_ylim()
    for year, label in events:
        ax.axvline(year, linestyle=':', linewidth=1)
        ax.text(year + .2, ymax, label, rotation=90, va='top', fontsize=8)
    ax.set_ylim(ymin, ymax)

    if lang == 'ptbr':
        ax.set_xlabel('Ano da primeira entrada persistente da cana')
        ax.set_ylabel('log₂ da razão de seleção')
        ax.set_title('Evolução anual da seleção topográfica — todas as faixas')
        stem_s = 'Figura_S2_evolucao_anual_selecao_topografica_todas_faixas'
    else:
        ax.set_xlabel('Year of first persistent sugarcane entry')
        ax.set_ylabel('log₂ selection ratio')
        ax.set_title('Annual topographic selection — all slope classes')
        stem_s = 'Figure_S2_annual_topographic_selection_all_classes'

    ax.legend(frameon=False, ncol=3, loc='lower left')
    add_panel_grid(ax)
    fig.tight_layout()
    save_lang_figure(fig, stem_s, lang)
    plt.show()

# Compatibilidade com diretório raiz, PT-BR.
fig, ax = plt.subplots(figsize=(11, 6.0))
for slope in plot_bins_main:
    g = annual_topo.loc[annual_topo['slope_bin'].eq(slope)].sort_values('year')
    ax.plot(g['year'], np.log2(g['selection_ratio']), marker='o', markersize=3, linewidth=1.6, label=SLOPE_LABELS_PTBR[slope])
ax.axhline(0, linestyle='--', linewidth=1, label='Uso proporcional à disponibilidade')
ymin, ymax = ax.get_ylim()
for year, label in [(2002, 'Lei 11.241/2002'), (2009, 'ZAE-Cana'), (2020, 'Pós-revogação')]:
    ax.axvline(year, linestyle=':', linewidth=1)
    ax.text(year + .2, ymax, label, rotation=90, va='top', fontsize=8)
ax.set_ylim(ymin, ymax)
ax.set_xlabel('Ano da primeira entrada persistente da cana')
ax.set_ylabel('log₂ da razão de seleção')
ax.set_title('Evolução anual da seleção topográfica da expansão canavieira')
ax.legend(frameon=False, ncol=2, loc='lower left')
add_panel_grid(ax)
fig.tight_layout()
save_figure(fig, 'Figura_4_evolucao_anual_selecao_topografica')
plt.close(fig)


In [ ]:

# ============================================================
# 4.4. TABELAS DA EXPANSÃO E DOS CONTRASTES
# ============================================================
expansion_period = entry_origin.copy()
expansion_period['years_in_period'] = (
    expansion_period['period'].map(PERIOD_YEARS)
)
expansion_period['annual_entry_ha'] = (
    expansion_period['all_persistent_entry_ha']
    / expansion_period['years_in_period']
)

expansion_period.to_csv(
    FINAL_TABLES / 'Tabela_2_expansao_por_periodo.csv',
    index=False,
    encoding='utf-8-sig'
)

period_topo.to_csv(
    FINAL_TABLES / 'Tabela_3_selecao_topografica_por_periodo.csv',
    index=False,
    encoding='utf-8-sig'
)

topo_contrasts.to_csv(
    FINAL_TABLES / 'Tabela_4_contrastes_topograficos_bootstrap.csv',
    index=False,
    encoding='utf-8-sig'
)

segmented_coef.to_csv(
    FINAL_DIAGNOSTICS / 'tendencias_segmentadas_coeficientes.csv',
    index=False,
    encoding='utf-8-sig'
)

segmented_slopes.to_csv(
    FINAL_DIAGNOSTICS / 'tendencias_segmentadas_inclinacoes.csv',
    index=False,
    encoding='utf-8-sig'
)

initial_stock.to_csv(
    FINAL_DIAGNOSTICS / 'estoque_inicial_declividade.csv',
    index=False,
    encoding='utf-8-sig'
)

display(expansion_period.round(4))
display(
    topo_contrasts[
        [
            'contrast', 'slope_label',
            'selection_ratio_earlier',
            'selection_ratio_later',
            'ratio_of_selection_ratios',
            'ratio_of_ratios_ci_low',
            'ratio_of_ratios_ci_high',
            'p_fdr_bh',
            'direction'
        ]
    ].round(4)
)



## 5. Uso anterior das novas áreas de cana

Este módulo é inteiramente local. Ele procura os oito CSVs já exportados pelo Earth Engine. Caso ainda não estejam todos no Drive, registra o módulo como pendente e segue para NEEA e vol%.


In [ ]:
# ============================================================
# 5.1. BASE LONGA DE USO ANTERIOR
#
# Duas rotas de entrada, em ordem de precedência:
#
#   (a) Base consolidada publicada — `previous_land_use_grid_year_long`
#       (.parquet ou .csv). É o produto direto da consolidação das oito
#       exportações do Google Earth Engine, preservado no pacote de
#       replicação. Rota canônica para reprodução.
#
#   (b) As oito exportações brutas do GEE, no padrão
#       `previous_land_use_AAAA_AAAA.csv`. Rota original de construção,
#       usada quando a base consolidada não está disponível.
#
# Em ambos os casos o resultado é `origin_long`, com estrutura idêntica.
# ============================================================

origin_long = None
origin_source = None

# ---- rota (a): base consolidada publicada -------------------------------
_consolidado = None
for _cand in [
    ORIGIN_DIR / 'previous_land_use_grid_year_long.parquet',
    ORIGIN_DIR / 'previous_land_use_grid_year_long.csv',
    OUT_DIR / 'previous_land_use_grid_year_long.parquet',
    OUT_DIR / 'previous_land_use_grid_year_long.csv',
]:
    if _cand.exists():
        _consolidado = _cand
        break

if _consolidado is not None:
    if _consolidado.suffix == '.parquet':
        origin_long = pd.read_parquet(_consolidado)
    else:
        origin_long = pd.read_csv(_consolidado, low_memory=False)

    origin_cell_id = 'grid_id_used'
    origin_long['year'] = pd.to_numeric(origin_long['year'], errors='coerce')
    origin_long['entry_area_ha'] = pd.to_numeric(
        origin_long['entry_area_ha'], errors='coerce'
    ).fillna(0)

    # Rótulos em inglês são rederivados: podem não constar do arquivo salvo.
    origin_long['origin_label'] = origin_long['origin_code'].map(ORIGIN_LABELS_PTBR)
    origin_long['origin_label_en'] = origin_long['origin_code'].map(ORIGIN_LABELS_EN)
    origin_long['period_label'] = origin_long['period'].map(PERIOD_LABELS_PTBR)
    origin_long['period_label_en'] = origin_long['period'].map(PERIOD_LABELS_EN)

    origin_source = f'base consolidada publicada ({_consolidado.name})'
    print(f'Uso anterior: {origin_source}')
    print(f'  {len(origin_long):,} linhas | '
          f'{origin_long["grid_id_used"].nunique():,} células da grade | '
          f'anos {int(origin_long["year"].min())}-{int(origin_long["year"].max())}')

# ---- rota (b): oito exportações brutas do GEE ---------------------------
else:
    origin_export_candidates = sorted([
        p for p in ORIGIN_DIR.rglob('previous_land_use_*.csv')
        if re.fullmatch(r'previous_land_use_\d{4}_\d{4}\.csv', p.name)
    ])

    if len(origin_export_candidates) == 8:
        raw_origin = pd.concat(
            [pd.read_csv(p, low_memory=False) for p in origin_export_candidates],
            ignore_index=True
        )

        # Identificador espacial estável: system:index varia entre exports.
        if 'grid_id' in raw_origin.columns:
            origin_cell_id = 'grid_id'
        elif 'id' in raw_origin.columns:
            origin_cell_id = 'id'
        elif 'cell_id' in raw_origin.columns:
            origin_cell_id = 'cell_id'
        elif 'fid' in raw_origin.columns:
            origin_cell_id = 'fid'
        else:
            raw_origin['cell_id_generated'] = (
                raw_origin.groupby('year').cumcount().astype(str)
            )
            origin_cell_id = 'cell_id_generated'

        print(f'Identificador espacial usado: {origin_cell_id}')

        origin_cols = [
            c for c in raw_origin.columns
            if c.startswith('origin_') and c.endswith('_ha')
        ]
        value_map = {}
        for col in origin_cols:
            match = re.match(r'origin_(\d+)_', col)
            if match:
                value_map[col] = int(match.group(1))

        if len(value_map) != 7:
            raise ValueError(
                f'Esperava sete bandas de origem, encontrei {len(value_map)}: '
                f'{list(value_map)}'
            )

        origin_long = raw_origin.melt(
            id_vars=[origin_cell_id, 'year', 'period'],
            value_vars=list(value_map),
            var_name='origin_column',
            value_name='entry_area_ha'
        )
        origin_long = origin_long.rename(columns={origin_cell_id: 'grid_id_used'})
        origin_cell_id = 'grid_id_used'
        origin_long['year'] = pd.to_numeric(origin_long['year'], errors='coerce')
        origin_long['origin_code'] = origin_long['origin_column'].map(value_map)
        origin_long['origin_label'] = origin_long['origin_code'].map(ORIGIN_LABELS_PTBR)
        origin_long['origin_label_en'] = origin_long['origin_code'].map(ORIGIN_LABELS_EN)
        origin_long['entry_area_ha'] = pd.to_numeric(
            origin_long['entry_area_ha'], errors='coerce'
        ).fillna(0)
        origin_long['period_label'] = origin_long['period'].map(PERIOD_LABELS_PTBR)
        origin_long['period_label_en'] = origin_long['period'].map(PERIOD_LABELS_EN)

        origin_source = 'oito exportacoes brutas do GEE'
        print(f'Uso anterior: {origin_source}')
    else:
        print(
            f'Modulo de uso anterior indisponivel: base consolidada ausente e '
            f'{len(origin_export_candidates)}/8 exportacoes brutas encontradas.'
        )

origin_ready = origin_long is not None

if not origin_ready:
    print('Os demais modulos podem ser executados normalmente.')
else:
    # Soma das sete classes mapeadas.
    origin_summary_mapped = (
        origin_long
        .groupby(
            ['period', 'period_label', 'period_label_en', 'origin_code', 'origin_label', 'origin_label_en'],
            as_index=False
        )['entry_area_ha']
        .sum()
    )

    mapped_totals = (
        origin_summary_mapped
        .groupby('period', as_index=False)['entry_area_ha']
        .sum()
        .rename(columns={'entry_area_ha': 'mapped_origin_area_ha'})
    )

    # Denominador compatível com a T2: área total de entrada persistente por período.
    # Quando a T2 contém uma pequena diferença em relação à soma das 7 classes, adicionamos
    # uma classe residual para fechar o denominador sem mascarar a diferença.
    if 'expansion_period' in globals() and 'all_persistent_entry_ha' in expansion_period.columns:
        period_totals = expansion_period[['period', 'all_persistent_entry_ha']].drop_duplicates('period')
        period_totals = period_totals.rename(columns={'all_persistent_entry_ha': 'period_entry_total_ha'})
    else:
        period_totals = mapped_totals.rename(columns={'mapped_origin_area_ha': 'period_entry_total_ha'})

    origin_summary = origin_summary_mapped.merge(mapped_totals, on='period', how='left')
    origin_summary = origin_summary.merge(period_totals, on='period', how='left')

    residual = period_totals.merge(mapped_totals, on='period', how='left')
    residual['mapped_origin_area_ha'] = residual['mapped_origin_area_ha'].fillna(0)
    residual['entry_area_ha'] = (
        residual['period_entry_total_ha'] - residual['mapped_origin_area_ha']
    ).clip(lower=0)

    residual_rows = residual.loc[residual['entry_area_ha'].gt(1e-6)].copy()
    if not residual_rows.empty:
        residual_rows['period_label'] = residual_rows['period'].map(PERIOD_LABELS_PTBR)
        residual_rows['period_label_en'] = residual_rows['period'].map(PERIOD_LABELS_EN)
        residual_rows['origin_code'] = 99
        residual_rows['origin_label'] = 'Não classificado / sem dado'
        residual_rows['origin_label_en'] = 'Unclassified / no data'
        residual_rows = residual_rows[[
            'period', 'period_label', 'period_label_en', 'origin_code',
            'origin_label', 'origin_label_en', 'entry_area_ha',
            'mapped_origin_area_ha', 'period_entry_total_ha'
        ]]
        origin_summary = pd.concat([origin_summary, residual_rows], ignore_index=True)

    origin_summary['unclassified_gap_ha'] = (
        origin_summary['period_entry_total_ha'] - origin_summary['mapped_origin_area_ha']
    ).clip(lower=0)
    origin_summary['origin_share'] = np.divide(
        origin_summary['entry_area_ha'],
        origin_summary['period_entry_total_ha'],
        out=np.zeros(len(origin_summary), dtype=float),
        where=origin_summary['period_entry_total_ha'].to_numpy() > 0
    )
    origin_summary['mapped_origin_share'] = np.divide(
        origin_summary['entry_area_ha'],
        origin_summary['mapped_origin_area_ha'],
        out=np.zeros(len(origin_summary), dtype=float),
        where=origin_summary['mapped_origin_area_ha'].to_numpy() > 0
    )
    origin_summary.loc[origin_summary['origin_code'].eq(99), 'mapped_origin_share'] = np.nan

    origin_summary['period_order'] = origin_summary['period'].map({p: i for i, p in enumerate(PERIOD_ORDER)})
    origin_summary['origin_order'] = origin_summary['origin_code'].replace({99: 999})
    origin_summary = origin_summary.sort_values(['period_order', 'origin_order']).reset_index(drop=True)
    origin_summary['denominator_note'] = (
        'origin_share uses total persistent-entry area from T2; '
        'mapped_origin_share uses only the seven mapped origin classes; '
        'origin_code 99 closes the residual gap when present.'
    )

    origin_long.to_parquet(
        ORIGIN_DIR / 'previous_land_use_grid_year_long.parquet',
        index=False
    )

    origin_summary.to_csv(
        ORIGIN_DIR / 'previous_land_use_period_summary.csv',
        index=False,
        encoding='utf-8-sig'
    )

    origin_summary.to_csv(
        FINAL_TABLES / 'Tabela_5_uso_anterior_por_periodo.csv',
        index=False,
        encoding='utf-8-sig'
    )

    origin_summary.rename(columns={
        'period_label': 'período',
        'origin_label': 'uso_anterior',
        'entry_area_ha': 'área_ha',
        'origin_share': 'participação_total_expansão',
        'mapped_origin_share': 'participação_classes_mapeadas'
    }).to_csv(
        FINAL_TABLES_PTBR / 'Tabela_5_uso_anterior_por_periodo_ptbr.csv',
        index=False,
        encoding='utf-8-sig'
    )

    origin_summary.rename(columns={
        'period_label_en': 'period',
        'origin_label_en': 'previous_land_use',
        'entry_area_ha': 'area_ha',
        'origin_share': 'share_of_total_expansion',
        'mapped_origin_share': 'share_of_mapped_classes'
    })[
        [
            'period', 'origin_code', 'previous_land_use', 'area_ha',
            'share_of_total_expansion', 'share_of_mapped_classes',
            'period_entry_total_ha', 'mapped_origin_area_ha', 'unclassified_gap_ha'
        ]
    ].to_csv(
        FINAL_TABLES_EN / 'Table_5_previous_land_use_by_period_en.csv',
        index=False,
        encoding='utf-8-sig'
    )

    print(
        'Uso anterior consolidado:',
        origin_long.shape,
        '| célula:', origin_cell_id
    )
    display(origin_summary)


In [ ]:

# ============================================================
# 5.2. BOOTSTRAP ESPACIAL VETORIZADO DO USO ANTERIOR
# ============================================================
if origin_ready:
    cells_unique = sorted(
        origin_long[origin_cell_id].dropna().unique()
    )
    origins = sorted(ORIGIN_LABELS_PTBR)
    n_cells = len(cells_unique)

    grid = (
        origin_long
        .groupby(
            [origin_cell_id, 'period', 'origin_code'],
            as_index=False
        )['entry_area_ha']
        .sum()
    )

    full_index = pd.MultiIndex.from_product(
        [cells_unique, PERIOD_ORDER, origins],
        names=[origin_cell_id, 'period', 'origin_code']
    )

    complete = (
        grid
        .set_index([origin_cell_id, 'period', 'origin_code'])
        .reindex(full_index)
        .fillna(0)
        .reset_index()
    )

    area_array = (
        complete['entry_area_ha']
        .to_numpy()
        .reshape(n_cells, len(PERIOD_ORDER), len(origins))
    )

    rng = np.random.default_rng(CFG['origin_seed'])

    contrast_pairs = [
        (0, 1, 'Pós-Lei versus pré-Lei', 'Post-law versus pre-law'),
        (1, 2, 'ZAE versus pós-Lei/pré-ZAE', 'ZAE versus post-law/pre-ZAE'),
        (2, 3, 'Pós-revogação versus ZAE', 'Post-revocation versus ZAE'),
    ]

    draws = []

    for b in range(CFG['n_boot_origin']):
        sampled = rng.integers(0, n_cells, size=n_cells)
        totals = area_array[sampled].sum(axis=0)

        period_totals = totals.sum(axis=1, keepdims=True)
        shares = np.divide(
            totals,
            period_totals,
            out=np.zeros_like(totals),
            where=period_totals > 0
        )

        for earlier, later, label_pt, label_en in contrast_pairs:
            for oi, code in enumerate(origins):
                earlier_share = shares[earlier, oi]
                later_share = shares[later, oi]

                draws.append({
                    'bootstrap': b,
                    'contrast': label_pt,
                    'contrast_en': label_en,
                    'origin_code': code,
                    'origin_label': ORIGIN_LABELS_PTBR[code],
                    'origin_label_en': ORIGIN_LABELS_EN[code],
                    'difference_pp': 100 * (
                        later_share - earlier_share
                    ),
                    'ratio_of_shares': (
                        later_share / earlier_share
                        if earlier_share > 0 else np.nan
                    )
                })

        if (b + 1) % 250 == 0:
            print(
                f'Bootstrap do uso anterior: '
                f'{b + 1}/{CFG["n_boot_origin"]}'
            )

    origin_boot = pd.DataFrame(draws)

    origin_contrasts = (
        origin_boot
        .groupby(['contrast', 'contrast_en', 'origin_code', 'origin_label', 'origin_label_en'])
        .agg(
            difference_pp=('difference_pp', 'mean'),
            difference_pp_ci_low=(
                'difference_pp',
                lambda x: np.quantile(x, .025)
            ),
            difference_pp_ci_high=(
                'difference_pp',
                lambda x: np.quantile(x, .975)
            ),
            ratio_of_shares=('ratio_of_shares', 'mean'),
            ratio_ci_low=(
                'ratio_of_shares',
                lambda x: np.nanquantile(x, .025)
            ),
            ratio_ci_high=(
                'ratio_of_shares',
                lambda x: np.nanquantile(x, .975)
            ),
        )
        .reset_index()
    )

    origin_contrasts['difference_supported'] = ~(
        (origin_contrasts['difference_pp_ci_low'] <= 0)
        & (origin_contrasts['difference_pp_ci_high'] >= 0)
    )

    origin_contrasts.to_csv(
        FINAL_TABLES / 'Tabela_6_contrastes_uso_anterior_bootstrap.csv',
        index=False,
        encoding='utf-8-sig'
    )

    origin_contrasts.rename(columns={
        'contrast': 'contraste',
        'origin_label': 'uso_anterior',
        'difference_pp': 'diferença_pp',
        'ratio_of_shares': 'razão_participações',
        'difference_supported': 'diferença_conclusiva'
    }).to_csv(
        FINAL_TABLES_PTBR / 'Tabela_6_contrastes_uso_anterior_bootstrap_ptbr.csv',
        index=False,
        encoding='utf-8-sig'
    )

    origin_contrasts.rename(columns={
        'contrast_en': 'contrast',
        'origin_label_en': 'previous_land_use',
        'difference_supported': 'supported_difference'
    })[
        [
            'contrast', 'origin_code', 'previous_land_use',
            'difference_pp', 'difference_pp_ci_low', 'difference_pp_ci_high',
            'ratio_of_shares', 'ratio_ci_low', 'ratio_ci_high',
            'supported_difference'
        ]
    ].to_csv(
        FINAL_TABLES_EN / 'Table_6_previous_land_use_bootstrap_contrasts_en.csv',
        index=False,
        encoding='utf-8-sig'
    )

    if CFG['save_origin_draws']:
        origin_boot.to_parquet(
            ORIGIN_DIR / 'previous_land_use_bootstrap_draws.parquet',
            index=False
        )

    display(origin_contrasts.round(4))


In [ ]:
# ============================================================
# 5.3. FIGURAS DO USO ANTERIOR
# ============================================================
if origin_ready:
    origin_order_plot = [1, 2, 3, 4, 5, 6, 7]
    if origin_summary['origin_code'].eq(99).any():
        origin_order_plot.append(99)

    origin_labels_pt = {**ORIGIN_LABELS_PTBR, 99: 'Não classificado / sem dado'}
    origin_labels_en = {**ORIGIN_LABELS_EN, 99: 'Unclassified / no data'}

    for lang in LANGS:
        if lang == 'ptbr':
            period_labels = PERIOD_LABELS_PTBR
            origin_labels = origin_labels_pt
            title_heat = 'Composição do uso anterior por período institucional'
            title_bar = 'Uso anterior das novas áreas de cana-de-açúcar'
            cbar_label = 'Participação no total da expansão (%)'
            y_label = 'Participação nas novas entradas persistentes'
            legend_title = 'Uso anterior'
            heat_stem = 'Figura_5_uso_anterior_heatmap'
            bar_stem = 'Figura_S1_uso_anterior_barras_empilhadas'
        else:
            period_labels = PERIOD_LABELS_EN
            origin_labels = origin_labels_en
            title_heat = 'Previous land-use composition by institutional period'
            title_bar = 'Previous land use of new sugarcane areas'
            cbar_label = 'Share of total expansion (%)'
            y_label = 'Share of new persistent entries'
            legend_title = 'Previous land use'
            heat_stem = 'Figure_5_previous_land_use_heatmap'
            bar_stem = 'Figure_S1_previous_land_use_stacked_bars'

        tmp = origin_summary.copy()
        tmp['period_display'] = tmp['period'].map(period_labels)
        tmp['origin_display'] = tmp['origin_code'].map(origin_labels)

        plot_origin = (
            tmp
            .pivot(index='period', columns='origin_code', values='origin_share')
            .reindex(PERIOD_ORDER)
            .reindex(columns=origin_order_plot)
            .fillna(0)
        )
        plot_origin.columns = [origin_labels[c] for c in plot_origin.columns]
        plot_origin.index = [period_labels[p] for p in plot_origin.index]

        # Figura principal: heatmap.
        mat = plot_origin.T * 100
        fig, ax = plt.subplots(figsize=(11.2, 5.8))
        image = ax.imshow(mat.values, aspect='auto', cmap='viridis')
        ax.set_xticks(range(len(mat.columns)))
        ax.set_xticklabels(mat.columns, rotation=20, ha='right')
        ax.set_yticks(range(len(mat.index)))
        ax.set_yticklabels(mat.index)

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                ax.text(j, i, f'{mat.iloc[i, j]:.1f}%', ha='center', va='center', fontsize=8, color='black')

        ax.set_title(title_heat)
        fig.colorbar(image, ax=ax, label=cbar_label)
        fig.tight_layout()
        save_lang_figure(fig, heat_stem, lang)
        plt.show()

        # Suplemento: barras empilhadas.
        fig, ax = plt.subplots(figsize=(12, 6.5))
        plot_origin.plot(kind='bar', stacked=True, ax=ax, width=.75)
        ax.set_ylabel(y_label)
        ax.set_title(title_bar)
        ax.set_xlabel('')
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
        ax.legend(title=legend_title, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
        plt.xticks(rotation=15, ha='right')
        add_panel_grid(ax)
        fig.tight_layout()
        save_lang_figure(fig, bar_stem, lang)
        plt.show()

    # Compatibilidade com diretório raiz: heatmap como Figura 5.
    tmp = origin_summary.copy()
    plot_origin_pt = (
        tmp
        .pivot(index='period', columns='origin_code', values='origin_share')
        .reindex(PERIOD_ORDER)
        .reindex(columns=origin_order_plot)
        .fillna(0)
    )
    plot_origin_pt.columns = [origin_labels_pt[c] for c in plot_origin_pt.columns]
    plot_origin_pt.index = [PERIOD_LABELS_PTBR[p] for p in plot_origin_pt.index]
    mat = plot_origin_pt.T * 100
    fig, ax = plt.subplots(figsize=(11.2, 5.8))
    image = ax.imshow(mat.values, aspect='auto', cmap='viridis')
    ax.set_xticks(range(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=20, ha='right')
    ax.set_yticks(range(len(mat.index)))
    ax.set_yticklabels(mat.index)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f'{mat.iloc[i, j]:.1f}%', ha='center', va='center', fontsize=8, color='black')
    ax.set_title('Composição do uso anterior por período institucional')
    fig.colorbar(image, ax=ax, label='Participação no total da expansão (%)')
    fig.tight_layout()
    save_figure(fig, 'Figura_5_uso_anterior_heatmap')
    plt.close(fig)



## 5.4. Uso anterior por unidade e robustez de buffers

Esta seção cria uma camada exploratória para testar se a composição do uso anterior das novas áreas de cana está associada à ΔNEEA.

Quando existir um cache exato por unidade e buffer, o notebook o utiliza. Caso contrário, pode calcular uma **aproximação espacial** a partir dos CSVs de uso anterior por grade, ponderando cada célula pela interseção com buffers circulares das unidades. Essa aproximação é útil como teste exploratório, mas deve ser descrita como tal, pois assume distribuição homogênea das novas entradas dentro de cada célula de 50 km.


In [ ]:

# ============================================================
# 5.4. USO ANTERIOR POR UNIDADE E BUFFER
# ============================================================
unit_origin_all_buffers = pd.DataFrame()
unit_origin_missing_buffers = []

def find_unit_origin_cache(buffer_km):
    return first_existing([
        ORIGIN_DIR / f'previous_land_use_units_{buffer_km}km.csv',
        ORIGIN_DIR / f'unit_previous_land_use_{buffer_km}km.csv',
        FINAL_ROBUSTNESS / f'previous_land_use_units_{buffer_km}km.csv',
    ], required=False)


# ------------------------------------------------------------
# Coordenadas das usinas
# ------------------------------------------------------------
# A base pode trazer longitude/latitude ou coordenadas projetadas em metros.
# Esta rotina infere o CRS testando candidatos e escolhendo aquele que projeta
# a maior parte dos pontos para dentro do envelope do Brasil/Centro-Sul.

UNIT_COORD_CRS_USED = None
UNITS_GDF_CACHE = None


def _candidate_unit_crs_list():
    candidates = []

    # Permite forçar manualmente, se necessário:
    # CFG['unit_coord_crs'] = 'EPSG:5880'
    forced = CFG.get('unit_coord_crs', None)
    if forced:
        candidates.append(str(forced))

    # Geográficas e projetadas plausíveis para Brasil/Centro-Sul.
    candidates.extend([
        'EPSG:4326',   # WGS84 lon/lat
        'EPSG:4674',   # SIRGAS 2000 lon/lat
        'EPSG:5880',   # SIRGAS 2000 / Brazil Polyconic, metros
        'EPSG:3857',   # Web Mercator, metros
        'EPSG:5641',   # Brazil Mercator/SIRGAS, metros
        'EPSG:31981', 'EPSG:31982', 'EPSG:31983',
        'EPSG:31984', 'EPSG:31985', 'EPSG:31986',
    ])

    # Remove duplicatas preservando ordem.
    return list(dict.fromkeys(candidates))


def _score_crs_as_brazil(points_gdf):
    try:
        g4326 = points_gdf.to_crs('EPSG:4326')
        lon = g4326.geometry.x
        lat = g4326.geometry.y
        valid = (
            lon.between(-76, -28)
            & lat.between(-36, 8)
            & np.isfinite(lon)
            & np.isfinite(lat)
        )
        return float(valid.mean()) if len(valid) else 0.0
    except Exception:
        return 0.0


def build_points_geodataframe(df, id_cols=None, label='pontos'):
    """Cria GeoDataFrame de pontos inferindo CRS das colunas x_col/y_col."""
    if id_cols is None:
        id_cols = []

    cols = [c for c in id_cols if c in df.columns] + [x_col, y_col]
    pts = df[cols].copy()
    pts[x_col] = pd.to_numeric(pts[x_col], errors='coerce')
    pts[y_col] = pd.to_numeric(pts[y_col], errors='coerce')
    pts = pts.dropna(subset=[x_col, y_col]).copy()

    if pts.empty:
        raise ValueError(f'Não há coordenadas válidas para {label}.')

    # Se estiver em lon/lat, testa primeiro os CRSs geográficos.
    x_is_lon = pts[x_col].between(-180, 180).all()
    y_is_lat = pts[y_col].between(-90, 90).all()

    candidates = _candidate_unit_crs_list()
    if x_is_lon and y_is_lat:
        candidates = ['EPSG:4326', 'EPSG:4674'] + [c for c in candidates if c not in ['EPSG:4326', 'EPSG:4674']]

    scores = []
    for crs in candidates:
        try:
            test = gpd.GeoDataFrame(
                pts.copy(),
                geometry=gpd.points_from_xy(pts[x_col], pts[y_col]),
                crs=crs
            )
            score = _score_crs_as_brazil(test)
            scores.append((crs, score))
        except Exception:
            scores.append((crs, 0.0))

    best_crs, best_score = max(scores, key=lambda z: z[1])

    if best_score < 0.80:
        score_txt = ', '.join([f'{crs}: {score:.2f}' for crs, score in scores])
        raise ValueError(
            f'Não foi possível inferir com segurança o CRS das coordenadas de {label}.\n'
            f'Colunas usadas: x_col={x_col}, y_col={y_col}.\n'
            f'Intervalos: {x_col}=({pts[x_col].min():.3f}, {pts[x_col].max():.3f}); '
            f'{y_col}=({pts[y_col].min():.3f}, {pts[y_col].max():.3f}).\n'
            f'Pontuação dos CRSs candidatos: {score_txt}.\n'
            "Defina manualmente, por exemplo: CFG['unit_coord_crs'] = 'EPSG:5880'."
        )

    gdf = gpd.GeoDataFrame(
        pts,
        geometry=gpd.points_from_xy(pts[x_col], pts[y_col]),
        crs=best_crs
    )
    gdf['coord_crs_used'] = best_crs
    return gdf


def build_units_geodataframe():
    """GeoDataFrame de usinas, uma linha por CNPJ, com CRS inferido automaticamente."""
    global UNIT_COORD_CRS_USED, UNITS_GDF_CACHE

    if UNITS_GDF_CACHE is not None:
        return UNITS_GDF_CACHE.copy()

    unit_cols = ['cnpj_clean', x_col, y_col]
    keep = [c for c in ['emissor', 'cidade', uf_col] if c in main.columns]
    units = (
        main[unit_cols + keep]
        .dropna(subset=[x_col, y_col])
        .drop_duplicates('cnpj_clean')
        .copy()
    )

    units_gdf = build_points_geodataframe(
        units,
        id_cols=['cnpj_clean'] + keep,
        label='usinas'
    )

    UNIT_COORD_CRS_USED = units_gdf['coord_crs_used'].iloc[0]
    print(f'CRS inferido para coordenadas das usinas: {UNIT_COORD_CRS_USED}')

    # Diagnóstico salvo para auditoria.
    coord_diag = pd.DataFrame({
        'x_col': [x_col],
        'y_col': [y_col],
        'x_min': [units[x_col].min()],
        'x_max': [units[x_col].max()],
        'y_min': [units[y_col].min()],
        'y_max': [units[y_col].max()],
        'crs_used': [UNIT_COORD_CRS_USED],
        'n_units': [len(units_gdf)],
    })
    coord_diag.to_csv(
        FINAL_DIAGNOSTICS / 'diagnostico_crs_coordenadas_usinas.csv',
        index=False,
        encoding='utf-8-sig'
    )

    UNITS_GDF_CACHE = units_gdf.copy()
    return units_gdf.copy()


if origin_ready:
    for b in CFG['buffer_candidates_km']:
        cache = find_unit_origin_cache(b)

        if cache is not None:
            tmp = read_csv(cache, dtype={'cnpj_clean': 'string'})
            tmp['cnpj_clean'] = normalize_cnpj(tmp['cnpj_clean'])
            tmp['buffer_km'] = b
            if 'source_method' not in tmp.columns:
                tmp['source_method'] = 'exact_or_cached'
            unit_origin_all_buffers = pd.concat(
                [unit_origin_all_buffers, tmp],
                ignore_index=True
            )
            continue

        if not CFG['run_unit_origin_approximation_from_grid']:
            unit_origin_missing_buffers.append(b)
            continue

        if '.geo' not in raw_origin.columns:
            print(
                f'Buffer {b} km: não há coluna .geo nos CSVs de uso anterior; '
                'não foi possível aproximar por unidade.'
            )
            unit_origin_missing_buffers.append(b)
            continue

        print(f'Calculando aproximação de uso anterior por unidade — buffer {b} km...')

        raw_id_col = 'grid_id' if 'grid_id' in raw_origin.columns else (
            'id' if 'id' in raw_origin.columns else None
        )
        if raw_id_col is None:
            print(f'Buffer {b} km: não há grid_id/id na tabela bruta.')
            unit_origin_missing_buffers.append(b)
            continue

        # Geometria da grade: uma geometria por célula.
        grid_geom = (
            raw_origin[[raw_id_col, '.geo']]
            .drop_duplicates()
            .rename(columns={raw_id_col: 'grid_id_used'})
            .copy()
        )

        grid_geom['geometry'] = grid_geom['.geo'].apply(lambda x: shape(json.loads(x)))

        grid_gdf = gpd.GeoDataFrame(
            grid_geom[['grid_id_used', 'geometry']],
            geometry='geometry',
            crs='EPSG:4326'
        ).to_crs('EPSG:5880')
        grid_gdf['cell_area_m2'] = grid_gdf.geometry.area

        # Agregar origem dentro da janela pré-RenovaBio usada nos modelos.
        unit_origin_source = origin_long.loc[
            origin_long['year'].between(
                CFG['origin_model_year_min'],
                CFG['origin_model_year_max']
            )
        ].copy()

        grid_origin = (
            unit_origin_source
            .groupby(['grid_id_used', 'origin_code'], as_index=False)['entry_area_ha']
            .sum()
        )

        units_gdf = build_units_geodataframe().to_crs('EPSG:5880')
        buffers = units_gdf[['cnpj_clean', 'geometry']].copy()
        buffers['geometry'] = buffers.geometry.buffer(b * 1000)

        intersections = gpd.overlay(
            buffers,
            grid_gdf[['grid_id_used', 'cell_area_m2', 'geometry']],
            how='intersection',
            keep_geom_type=False
        )

        if intersections.empty:
            print(f'Buffer {b} km: nenhuma interseção entre unidades e grade.')
            unit_origin_missing_buffers.append(b)
            continue

        intersections['intersection_area_m2'] = intersections.geometry.area
        intersections['cell_weight'] = (
            intersections['intersection_area_m2'] / intersections['cell_area_m2']
        ).clip(lower=0, upper=1)

        weighted = intersections[
            ['cnpj_clean', 'grid_id_used', 'cell_weight']
        ].merge(
            grid_origin,
            on='grid_id_used',
            how='left'
        )

        weighted['entry_area_ha'] = weighted['entry_area_ha'].fillna(0)
        weighted['weighted_entry_area_ha'] = (
            weighted['entry_area_ha'] * weighted['cell_weight']
        )

        unit_origin = (
            weighted
            .groupby(['cnpj_clean', 'origin_code'], as_index=False)['weighted_entry_area_ha']
            .sum()
        )

        unit_origin['buffer_km'] = b
        unit_origin['origin_label'] = unit_origin['origin_code'].map(ORIGIN_LABELS_PTBR)
        unit_origin['origin_label_en'] = unit_origin['origin_code'].map(ORIGIN_LABELS_EN)

        totals = (
            unit_origin
            .groupby('cnpj_clean', as_index=False)['weighted_entry_area_ha']
            .sum()
            .rename(columns={'weighted_entry_area_ha': 'prior_entry_area_total_ha'})
        )

        wide = (
            unit_origin
            .pivot_table(
                index=['cnpj_clean', 'buffer_km'],
                columns='origin_code',
                values='weighted_entry_area_ha',
                aggfunc='sum',
                fill_value=0
            )
            .reset_index()
        )

        for code in sorted(ORIGIN_LABELS_EN):
            if code not in wide.columns:
                wide[code] = 0

        wide = wide.merge(totals, on='cnpj_clean', how='left')

        for code, label in ORIGIN_LABELS_EN.items():
            clean = (
                label.lower()
                .replace(' ', '_')
                .replace('-', '_')
            )
            wide[f'prior_{clean}_area_ha'] = wide[code]
            wide[f'prior_{clean}_share'] = np.divide(
                wide[code],
                wide['prior_entry_area_total_ha'],
                out=np.zeros(len(wide), dtype=float),
                where=wide['prior_entry_area_total_ha'].to_numpy() > 0
            )

        wide['prior_productive_share'] = (
            wide.get('prior_pasture_share', 0)
            + wide.get('prior_annual_agriculture_share', 0)
            + wide.get('prior_perennial_agriculture_share', 0)
            + wide.get('prior_agricultural_mosaic_share', 0)
            + wide.get('prior_forestry_share', 0)
        )
        wide['source_method'] = 'grid_intersection_approximation'
        wide['origin_year_window'] = (
            f'{CFG["origin_model_year_min"]}-{CFG["origin_model_year_max"]}'
        )

        drop_origin_numeric = [c for c in sorted(ORIGIN_LABELS_EN) if c in wide.columns]
        wide = wide.drop(columns=drop_origin_numeric)

        out_cache = ORIGIN_DIR / f'previous_land_use_units_{b}km.csv'
        wide.to_csv(out_cache, index=False, encoding='utf-8-sig')
        print('Cache salvo:', out_cache)

        unit_origin_all_buffers = pd.concat(
            [unit_origin_all_buffers, wide],
            ignore_index=True
        )

    if not unit_origin_all_buffers.empty:
        unit_origin_all_buffers['cnpj_clean'] = normalize_cnpj(
            unit_origin_all_buffers['cnpj_clean']
        )

        unit_origin_all_buffers.to_csv(
            FINAL_ROBUSTNESS / 'previous_land_use_units_all_buffers.csv',
            index=False,
            encoding='utf-8-sig'
        )

        # Incorpora o buffer ativo à amostra principal.
        unit_origin_active = unit_origin_all_buffers.loc[
            unit_origin_all_buffers['buffer_km'].eq(CFG['active_buffer_km'])
        ].copy()

        if not unit_origin_active.empty:
            prior_cols = [
                c for c in unit_origin_active.columns
                if c.startswith('prior_') or c in [
                    'cnpj_clean', 'buffer_km', 'source_method', 'origin_year_window'
                ]
            ]
            prior_cols = list(dict.fromkeys(prior_cols))

            main = main.drop(
                columns=[c for c in main.columns if c.startswith('prior_')],
                errors='ignore'
            ).merge(
                unit_origin_active[prior_cols].drop_duplicates('cnpj_clean'),
                on='cnpj_clean',
                how='left'
            )

            print(
                'Uso anterior por unidade incorporado à amostra principal:',
                unit_origin_active.shape
            )
        else:
            print(
                'AVISO: não há uso anterior por unidade para o buffer ativo '
                f'{CFG["active_buffer_km"]} km.'
            )

        display(unit_origin_all_buffers.head())
    else:
        print('Nenhuma camada de uso anterior por unidade foi criada ou encontrada.')

    if unit_origin_missing_buffers:
        pd.DataFrame({'buffer_km_missing_unit_origin': unit_origin_missing_buffers}).to_csv(
            FINAL_ROBUSTNESS / 'missing_unit_origin_buffers.csv',
            index=False,
            encoding='utf-8-sig'
        )
else:
    print('Módulo de uso anterior por unidade ignorado porque os 8 CSVs ainda não estão disponíveis.')



## 5.5. Fronteira aproximada por unidade e buffer

Esta seção cria caches multiescala para os modelos de robustez quando não existem estatísticas exatas de fronteira para 30, 50, 100 e 150 km. A aproximação usa as mesmas células de 50 km da Análise 1: para cada buffer de usina, as áreas anuais de novas entradas persistentes de cana são ponderadas pela fração da célula intersectada pelo buffer.

Use estes resultados como **robustez exploratória**. Eles não substituem uma recomputação exata pixel-a-pixel em GEE, mas permitem testar se o sinal fronteira–ΔNEEA depende fortemente do buffer de 25 km.

In [ ]:

# ============================================================
# 5.5. FRONTEIRA APROXIMADA POR UNIDADE E BUFFER
# ============================================================
unit_frontier_approx_all_buffers = pd.DataFrame()
unit_frontier_approx_missing_buffers = []


def _make_grid_gdf_from_raw_origin():
    """Reconstrói a geometria da grade dos CSVs de uso anterior."""
    if '.geo' not in raw_origin.columns:
        raise ValueError('A tabela raw_origin não possui coluna .geo.')

    raw_id_col = 'grid_id' if 'grid_id' in raw_origin.columns else (
        'id' if 'id' in raw_origin.columns else None
    )
    if raw_id_col is None:
        raise ValueError('A tabela raw_origin não possui grid_id ou id.')

    grid_geom = (
        raw_origin[[raw_id_col, '.geo']]
        .drop_duplicates()
        .rename(columns={raw_id_col: 'grid_id_used'})
        .copy()
    )
    grid_geom['geometry'] = grid_geom['.geo'].apply(lambda x: shape(json.loads(x)))

    grid_gdf = gpd.GeoDataFrame(
        grid_geom[['grid_id_used', 'geometry']],
        geometry='geometry',
        crs='EPSG:4326'
    ).to_crs('EPSG:5880')

    grid_gdf['cell_area_m2'] = grid_gdf.geometry.area
    return grid_gdf


def _panel_metadata_for_frontier_merge():
    cols = [
        c for c in [
            'cnpj_clean', 'emissor', 'cidade', 'uf', 'rota', 'rota_tag', 'cohort',
            'X', 'Y', 'longitude', 'latitude',
            'neea_first', 'neea_t2_2026', 'delta_neea',
            'vol_pct_first', 'vol_pct_t2_2026', 'delta_vol_pct'
        ]
        if c in panel.columns
    ]
    out = panel[cols].drop_duplicates('cnpj_clean').copy()
    out['cnpj_clean'] = normalize_cnpj(out['cnpj_clean'])
    return out


def build_analysis_long_from_frontier_stats(frontier_stats, buffer_km, source_label=None):
    """Cria base longa de modelos a partir de estatísticas de fronteira por CNPJ."""
    if frontier_stats is None or frontier_stats.empty:
        return pd.DataFrame()

    f = frontier_stats.copy()
    f['cnpj_clean'] = normalize_cnpj(f['cnpj_clean'])
    if 'schema' not in f.columns:
        f['schema'] = CFG['main_schema']
    if 'buffer_km' not in f.columns:
        f['buffer_km'] = buffer_km
    if source_label is not None and 'frontier_source_method' not in f.columns:
        f['frontier_source_method'] = source_label

    base = _panel_metadata_for_frontier_merge()
    merged = base.merge(
        f,
        on='cnpj_clean',
        how='inner',
        validate='one_to_many',
        suffixes=('', '_frontier')
    )

    neea = merged.copy()
    neea['outcome'] = 'delta_neea'
    vol = merged.copy()
    vol['outcome'] = 'delta_vol_pct'
    return pd.concat([neea, vol], ignore_index=True)


def build_approx_frontier_from_origin_grid(buffer_km, force=False):
    """Aproxima ano médio da fronteira e área nova de cana por buffer.

    A unidade de entrada é a grade de 50 km usada nos exports de uso anterior.
    A área anual de novas entradas persistentes em cada célula é ponderada pela
    fração da célula intersectada pelo buffer da usina.
    """
    out_frontier = ORIGIN_DIR / f'frontier_temporal_robustness_all_schemas_{buffer_km}km_approx_from_origin_grid.csv'
    out_long = ORIGIN_DIR / f'overlay_frontier_merged_long_{buffer_km}km_approx_from_origin_grid.csv'

    if out_frontier.exists() and out_long.exists() and not force:
        tmp = read_csv(out_frontier, dtype={'cnpj_clean': 'string'})
        tmp['cnpj_clean'] = normalize_cnpj(tmp['cnpj_clean'])
        return tmp

    if not origin_ready:
        raise RuntimeError('Os CSVs de uso anterior ainda não estão disponíveis.')

    grid_gdf = _make_grid_gdf_from_raw_origin()

    # Área anual total de novas entradas persistentes por célula.
    origin_window = origin_long.loc[
        origin_long['year'].between(
            CFG['origin_model_year_min'],
            CFG['origin_model_year_max']
        )
    ].copy()

    grid_year = (
        origin_window
        .groupby(['grid_id_used', 'year'], as_index=False)['entry_area_ha']
        .sum()
    )

    units_gdf = build_units_geodataframe().to_crs('EPSG:5880')
    buffers = units_gdf[['cnpj_clean', 'geometry']].copy()
    buffers['geometry'] = buffers.geometry.buffer(buffer_km * 1000)

    intersections = gpd.overlay(
        buffers,
        grid_gdf[['grid_id_used', 'cell_area_m2', 'geometry']],
        how='intersection',
        keep_geom_type=False
    )

    if intersections.empty:
        raise RuntimeError(f'Buffer {buffer_km} km: nenhuma interseção entre unidades e grade.')

    intersections['intersection_area_m2'] = intersections.geometry.area
    intersections['cell_weight'] = (
        intersections['intersection_area_m2'] / intersections['cell_area_m2']
    ).clip(lower=0, upper=1)

    weighted = intersections[
        ['cnpj_clean', 'grid_id_used', 'cell_weight']
    ].merge(
        grid_year,
        on='grid_id_used',
        how='left'
    )

    weighted['entry_area_ha'] = weighted['entry_area_ha'].fillna(0)
    weighted['weighted_entry_area_ha'] = weighted['entry_area_ha'] * weighted['cell_weight']

    agg = (
        weighted
        .groupby(['cnpj_clean', 'year'], as_index=False)['weighted_entry_area_ha']
        .sum()
    )

    totals = (
        agg
        .groupby('cnpj_clean', as_index=False)['weighted_entry_area_ha']
        .sum()
        .rename(columns={'weighted_entry_area_ha': 'cane_area_total_ha'})
    )

    # Média ponderada do ano de entrada. Se não houver nova cana no buffer, fica NaN.
    def _weighted_mean_year(g):
        area = g['weighted_entry_area_ha'].to_numpy(dtype=float)
        years = g['year'].to_numpy(dtype=float)
        total = area.sum()
        if total <= 0:
            return np.nan
        return np.average(years, weights=area)

    mean_year = (
        agg
        .groupby('cnpj_clean')
        .apply(_weighted_mean_year)
        .reset_index(name='mean_first_key_year')
    )

    first_last = (
        agg.loc[agg['weighted_entry_area_ha'].gt(0)]
        .groupby('cnpj_clean')
        .agg(
            first_entry_year=('year', 'min'),
            last_entry_year=('year', 'max'),
            n_entry_years=('year', 'nunique')
        )
        .reset_index()
    )

    frontier = (
        totals
        .merge(mean_year, on='cnpj_clean', how='left')
        .merge(first_last, on='cnpj_clean', how='left')
    )

    frontier['schema'] = CFG['main_schema']
    frontier['buffer_km'] = buffer_km
    frontier['frontier_source_method'] = 'approx_from_previous_land_use_grid'
    frontier['frontier_year_window'] = f'{CFG["origin_model_year_min"]}-{CFG["origin_model_year_max"]}'
    frontier['frontier_area_definition'] = 'persistent new sugarcane entries weighted by grid-cell overlap'

    # Preserva o nome principal usado nos modelos e adiciona um alias explícito.
    frontier['cane_area_total_ha_approx_from_origin'] = frontier['cane_area_total_ha']

    frontier.to_csv(out_frontier, index=False, encoding='utf-8-sig')

    long_df = build_analysis_long_from_frontier_stats(
        frontier,
        buffer_km=buffer_km,
        source_label='approx_from_previous_land_use_grid'
    )
    long_df.to_csv(out_long, index=False, encoding='utf-8-sig')

    print('Cache aproximado de fronteira salvo:', out_frontier)
    print('Base longa aproximada salva:', out_long)
    return frontier


if origin_ready and CFG.get('build_approx_frontier_from_origin_grid', True):
    approx_frames = []
    for b in CFG['buffer_candidates_km']:
        exact_available = (
            first_existing(model_data_candidates(b, include_generic=False), required=False) is not None
            or first_existing(frontier_candidates(b, include_generic=False), required=False) is not None
        )

        # Mantém o cache exato quando ele existe, mas ainda cria aproximação para buffers ausentes.
        if exact_available and CFG.get('prefer_exact_frontier_cache', True):
            print(f'Buffer {b} km: cache exato encontrado; aproximação não substitui a base principal.')
            continue

        try:
            print(f'Construindo fronteira aproximada — buffer {b} km...')
            approx_frames.append(build_approx_frontier_from_origin_grid(b))
        except Exception as exc:
            print(f'AVISO: falha ao construir fronteira aproximada para {b} km:', exc)
            unit_frontier_approx_missing_buffers.append({
                'buffer_km': b,
                'reason': str(exc)
            })

    # Também carrega aproximações já existentes, mesmo que criadas em execução anterior.
    for b in CFG['buffer_candidates_km']:
        cached = ORIGIN_DIR / f'frontier_temporal_robustness_all_schemas_{b}km_approx_from_origin_grid.csv'
        if cached.exists():
            tmp = read_csv(cached, dtype={'cnpj_clean': 'string'})
            tmp['cnpj_clean'] = normalize_cnpj(tmp['cnpj_clean'])
            approx_frames.append(tmp)

    if approx_frames:
        unit_frontier_approx_all_buffers = (
            pd.concat(approx_frames, ignore_index=True)
            .drop_duplicates(['cnpj_clean', 'schema', 'buffer_km'], keep='last')
        )
        unit_frontier_approx_all_buffers.to_csv(
            FINAL_ROBUSTNESS / 'frontier_approx_from_origin_grid_all_buffers.csv',
            index=False,
            encoding='utf-8-sig'
        )
        print('Fronteira aproximada disponível para buffers:', sorted(unit_frontier_approx_all_buffers['buffer_km'].unique()))
        display(unit_frontier_approx_all_buffers.head())

    if unit_frontier_approx_missing_buffers:
        pd.DataFrame(unit_frontier_approx_missing_buffers).to_csv(
            FINAL_ROBUSTNESS / 'missing_approx_frontier_buffers.csv',
            index=False,
            encoding='utf-8-sig'
        )
else:
    print('Fronteira aproximada por grade não foi executada.')



## 6. Geração histórica da fronteira, ΔNEEA e Δvol%

Os modelos principais usam o esquema pré-RenovaBio (1985–2015), efeitos estaduais e erros-padrão HC3. O ano médio da fronteira é centrado para melhorar a estabilidade numérica.


In [ ]:

# ============================================================
# 6.1. MODELOS PROGRESSIVOS PARA NEEA E VOL%
# ============================================================
MODEL_SPECS = {
    'delta_neea': {
        'baseline': 'neea_first',
        'label': 'ΔNEEA'
    },
    'delta_vol_pct': {
        'baseline': 'vol_pct_first',
        'label': 'Δvol%'
    },
}

# Uso anterior como teste exploratório.
# Mosaico agropecuário fica implicitamente como categoria de referência parcial.
PRIOR_USE_TERMS = [
    'prior_pasture_share',
    'prior_annual_agriculture_share',
    'prior_native_vegetation_share',
]
available_prior_terms = [c for c in PRIOR_USE_TERMS if c in main.columns]

print('Covariáveis de uso anterior disponíveis:', available_prior_terms)

model_rows = []
model_objects = {}
model_samples = []

for outcome_name, spec in MODEL_SPECS.items():
    sample = main.loc[
        main['outcome'].astype(str).eq(outcome_name)
    ].copy()

    baseline = spec['baseline']
    sample[f'{baseline}_c'] = (
        sample[baseline] - sample[baseline].mean()
    )

    formulas = {
        'M1 — fronteira': (
            f'{outcome_name} ~ frontier_year_c'
        ),
        'M2 — + nível inicial': (
            f'{outcome_name} ~ frontier_year_c + {baseline}_c'
        ),
        'M3 — + escala e UF': (
            f'{outcome_name} ~ frontier_year_c + {baseline}_c '
            f'+ log_cane_area + C({uf_col})'
        ),
    }

    if 'share_cane_slope_le12' in sample.columns:
        formulas['M4 — + topografia'] = (
            f'{outcome_name} ~ frontier_year_c + {baseline}_c '
            f'+ log_cane_area + share_cane_slope_le12 '
            f'+ C({uf_col})'
        )

    if available_prior_terms:
        prior_formula = ' + '.join(available_prior_terms)
        topo_term = (
            ' + share_cane_slope_le12'
            if 'share_cane_slope_le12' in sample.columns
            else ''
        )
        formulas['M5 — + uso anterior'] = (
            f'{outcome_name} ~ frontier_year_c + {baseline}_c '
            f'+ log_cane_area{topo_term} + {prior_formula} '
            f'+ C({uf_col})'
        )

    for model_name, formula in formulas.items():
        terms = [
            outcome_name, 'frontier_year_c',
            f'{baseline}_c', 'log_cane_area', uf_col
        ]
        if 'topografia' in model_name or 'uso anterior' in model_name:
            if 'share_cane_slope_le12' in sample.columns:
                terms.append('share_cane_slope_le12')
        if 'uso anterior' in model_name:
            terms.extend(available_prior_terms)

        terms = [c for c in terms if c in sample.columns]
        fit_sample = sample.dropna(subset=terms).copy()

        model_samples.append({
            'outcome': outcome_name,
            'model': model_name,
            'n': len(fit_sample),
            'formula': formula,
            'buffer_km': CFG['active_buffer_km'],
        })

        if len(fit_sample) < 20:
            print(f'AVISO: amostra pequena para {outcome_name} | {model_name}: n={len(fit_sample)}')
            continue

        model = smf.ols(
            formula,
            data=fit_sample
        ).fit(cov_type='HC3')

        model_objects[(outcome_name, model_name)] = (
            model, fit_sample
        )

        model_rows.append(
            tidy_model(model, outcome_name, model_name)
        )

model_table = pd.concat(model_rows, ignore_index=True)
model_sample_table = pd.DataFrame(model_samples)

model_table['buffer_km'] = CFG['active_buffer_km']

model_table.to_csv(
    FINAL_TABLES / 'Tabela_7_modelos_NEEA_vol.csv',
    index=False,
    encoding='utf-8-sig'
)

model_sample_table.to_csv(
    FINAL_DIAGNOSTICS / 'amostras_modelos_NEEA_vol.csv',
    index=False,
    encoding='utf-8-sig'
)

frontier_terms = model_table.loc[
    model_table['term'].eq('frontier_year_c')
].copy()

frontier_terms['effect_per_decade'] = (
    frontier_terms['coef'] * 10
)
frontier_terms['ci_low_per_decade'] = (
    frontier_terms['ci_low'] * 10
)
frontier_terms['ci_high_per_decade'] = (
    frontier_terms['ci_high'] * 10
)

frontier_terms.to_csv(
    FINAL_TABLES / 'Tabela_8_efeito_fronteira_por_decada.csv',
    index=False,
    encoding='utf-8-sig'
)

display(model_sample_table)
display(frontier_terms.round(4))



## 6.2. Robustez à escala do entorno territorial das unidades

Esta seção testa se o coeficiente da fronteira histórica permanece estável quando a caracterização territorial das unidades é recalculada em buffers de 25, 30, 50, 100 e 150 km. O buffer de 30 km é tratado como principal quando disponível; 100–150 km são interpretados como entorno regional ampliado, não como raio logístico de fornecimento.


In [ ]:

# ============================================================
# 6.2. ROBUSTEZ A BUFFERS DE 25/30/50/100/150 KM
# ============================================================
def _find_approx_model_cache(buffer_km):
    return first_existing([
        ORIGIN_DIR / f'overlay_frontier_merged_long_{buffer_km}km_approx_from_origin_grid.csv',
        FINAL_ROBUSTNESS / f'overlay_frontier_merged_long_{buffer_km}km_approx_from_origin_grid.csv',
    ], required=False)


def _find_approx_frontier_cache(buffer_km):
    return first_existing([
        ORIGIN_DIR / f'frontier_temporal_robustness_all_schemas_{buffer_km}km_approx_from_origin_grid.csv',
        FINAL_ROBUSTNESS / f'frontier_temporal_robustness_all_schemas_{buffer_km}km_approx_from_origin_grid.csv',
    ], required=False)


def load_analysis_for_buffer(buffer_km):
    """Carrega ou reconstrói base analítica para um buffer específico.

    Prioridade:
    1. Base/modelo exato do buffer;
    2. Estatísticas exatas de fronteira do buffer;
    3. Cache aproximado criado a partir da grade de uso anterior;
    4. Construção em memória a partir de unit_frontier_approx_all_buffers.
    """
    if buffer_km == CFG['active_buffer_km']:
        df = analysis_all.copy()
        if 'frontier_source_method' not in df.columns:
            df['frontier_source_method'] = 'exact_or_legacy_cache'
    else:
        model_path_b = first_existing(
            model_data_candidates(buffer_km, include_generic=False),
            required=False
        )
        frontier_path_b = first_existing(
            frontier_candidates(buffer_km, include_generic=False),
            required=False
        )
        approx_model_path_b = _find_approx_model_cache(buffer_km)
        approx_frontier_path_b = _find_approx_frontier_cache(buffer_km)

        source_method = None

        if model_path_b is not None:
            df = read_csv(model_path_b, dtype={'cnpj_clean': 'string'})
            df['cnpj_clean'] = normalize_cnpj(df['cnpj_clean'])
            source_method = 'exact_model_cache'
            if 'buffer_km' not in df.columns:
                df['buffer_km'] = buffer_km
        elif frontier_path_b is not None:
            frontier_b = read_csv(frontier_path_b, dtype={'cnpj_clean': 'string'})
            frontier_b['cnpj_clean'] = normalize_cnpj(frontier_b['cnpj_clean'])
            source_method = 'exact_frontier_cache'
            if 'buffer_km' not in frontier_b.columns:
                frontier_b['buffer_km'] = buffer_km
            df = build_analysis_long_from_frontier_stats(
                frontier_b,
                buffer_km=buffer_km,
                source_label=source_method
            )
        elif approx_model_path_b is not None:
            df = read_csv(approx_model_path_b, dtype={'cnpj_clean': 'string'})
            df['cnpj_clean'] = normalize_cnpj(df['cnpj_clean'])
            source_method = 'approx_model_cache_from_origin_grid'
            if 'buffer_km' not in df.columns:
                df['buffer_km'] = buffer_km
        elif approx_frontier_path_b is not None:
            frontier_b = read_csv(approx_frontier_path_b, dtype={'cnpj_clean': 'string'})
            frontier_b['cnpj_clean'] = normalize_cnpj(frontier_b['cnpj_clean'])
            source_method = 'approx_frontier_cache_from_origin_grid'
            df = build_analysis_long_from_frontier_stats(
                frontier_b,
                buffer_km=buffer_km,
                source_label=source_method
            )
        elif 'unit_frontier_approx_all_buffers' in globals() and not unit_frontier_approx_all_buffers.empty:
            frontier_b = unit_frontier_approx_all_buffers.loc[
                unit_frontier_approx_all_buffers['buffer_km'].eq(buffer_km)
            ].copy()
            if frontier_b.empty:
                return None
            source_method = 'approx_memory_from_origin_grid'
            df = build_analysis_long_from_frontier_stats(
                frontier_b,
                buffer_km=buffer_km,
                source_label=source_method
            )
        else:
            return None

        if 'frontier_source_method' not in df.columns:
            df['frontier_source_method'] = source_method or 'unknown'

    # Remover topografia potencialmente herdada e recombinar DEM do buffer.
    for c in ['share_cane_slope_le12', 'mean_cane_slope_pct', 'mean_cane_elevation_m']:
        if c in df.columns:
            df = df.drop(columns=c)

    dem_path_b = first_existing(
        dem_candidates(buffer_km, include_generic=False),
        required=False
    )

    # O DEM genérico é usado apenas para o buffer ativo/legado. Não copiar topografia de 25 km
    # para buffers maiores, porque isso daria falsa robustez multiescala.
    if dem_path_b is None and buffer_km == CFG['active_buffer_km']:
        dem_path_b = dem_path

    if dem_path_b is not None:
        dem_b = read_csv(dem_path_b, dtype={'cnpj_clean': 'string'})
        dem_b['cnpj_clean'] = normalize_cnpj(dem_b['cnpj_clean'])
        if 'buffer_km' not in dem_b.columns:
            dem_b['buffer_km'] = buffer_km

        dem_keep = [
            c for c in [
                'cnpj_clean', 'schema', 'buffer_km',
                'share_cane_slope_le12',
                'mean_cane_slope_pct',
                'mean_cane_elevation_m'
            ]
            if c in dem_b.columns
        ]

        dem_b = dem_b[dem_keep].drop_duplicates(['cnpj_clean', 'schema', 'buffer_km'])

        merge_keys = ['cnpj_clean', 'schema']
        if 'buffer_km' in df.columns and 'buffer_km' in dem_b.columns:
            merge_keys.append('buffer_km')

        df = df.merge(
            dem_b,
            on=merge_keys,
            how='left',
            validate='many_to_one'
        )
        df['topography_source_method'] = 'exact_dem_cache'
    else:
        df['topography_source_method'] = 'not_available_for_buffer'

    return df


def prepare_model_sample(df, outcome_name, buffer_km):
    df = df.copy()

    uf_b = 'uf_x' if 'uf_x' in df.columns else ('uf' if 'uf' in df.columns else 'uf_y')

    if {'longitude', 'latitude'}.issubset(df.columns):
        xb, yb = 'longitude', 'latitude'
    else:
        xb, yb = ('X', 'Y')

    for col in [
        'delta_neea', 'delta_vol_pct', 'neea_first', 'vol_pct_first',
        'mean_first_key_year', 'cane_area_total_ha',
        'share_cane_slope_le12', xb, yb
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df['log_cane_area'] = np.log1p(df['cane_area_total_ha'].clip(lower=0))

    sample = df.loc[
        df['schema'].astype(str).eq(CFG['main_schema'])
        & df['outcome'].astype(str).eq(outcome_name)
    ].copy()

    sample = (
        sample.sort_values(['outcome', 'cnpj_clean'])
        .drop_duplicates(['outcome', 'cnpj_clean'])
    )

    sample['frontier_year_c'] = (
        sample['mean_first_key_year']
        - sample['mean_first_key_year'].mean()
    )

    if 'unit_origin_all_buffers' in globals() and not unit_origin_all_buffers.empty:
        uo = unit_origin_all_buffers.loc[
            unit_origin_all_buffers['buffer_km'].eq(buffer_km)
        ].copy()

        if not uo.empty:
            prior_cols = [
                c for c in uo.columns
                if c.startswith('prior_') or c in ['cnpj_clean']
            ]
            prior_cols = list(dict.fromkeys(prior_cols))
            sample = sample.drop(
                columns=[c for c in sample.columns if c.startswith('prior_')],
                errors='ignore'
            ).merge(
                uo[prior_cols].drop_duplicates('cnpj_clean'),
                on='cnpj_clean',
                how='left'
            )

    return sample, uf_b


buffer_robust_rows = []
buffer_missing = []

for b in CFG['buffer_candidates_km']:
    df_b = load_analysis_for_buffer(b)

    if df_b is None:
        buffer_missing.append({
            'buffer_km': b,
            'reason': 'missing_exact_and_approx_frontier_or_model_cache'
        })
        continue

    frontier_source = (
        df_b['frontier_source_method'].dropna().iloc[0]
        if 'frontier_source_method' in df_b.columns and df_b['frontier_source_method'].notna().any()
        else 'unknown'
    )
    topo_source = (
        df_b['topography_source_method'].dropna().iloc[0]
        if 'topography_source_method' in df_b.columns and df_b['topography_source_method'].notna().any()
        else 'unknown'
    )

    for outcome_name, spec in MODEL_SPECS.items():
        sample_b, uf_b = prepare_model_sample(df_b, outcome_name, b)

        baseline = spec['baseline']
        if baseline not in sample_b.columns:
            continue

        sample_b[f'{baseline}_c'] = sample_b[baseline] - sample_b[baseline].mean()

        model_formulas = {}

        if CFG.get('run_m3_in_buffer_robustness', True):
            model_formulas['M3 — + escala e UF'] = (
                f'{outcome_name} ~ frontier_year_c + {baseline}_c '
                f'+ log_cane_area + C({uf_b})'
            )

        has_topography = (
            'share_cane_slope_le12' in sample_b.columns
            and sample_b['share_cane_slope_le12'].notna().sum() >= 20
        )

        if has_topography:
            model_formulas['M4 — + topografia'] = (
                f'{outcome_name} ~ frontier_year_c + {baseline}_c '
                f'+ log_cane_area + share_cane_slope_le12 + C({uf_b})'
            )

        prior_terms_b = [c for c in PRIOR_USE_TERMS if c in sample_b.columns]
        if prior_terms_b:
            prior_formula = ' + '.join(prior_terms_b)
            topo_term = ' + share_cane_slope_le12' if has_topography else ''
            model_name = 'M5 — + uso anterior' if has_topography else 'M5b — + uso anterior sem topografia'
            model_formulas[model_name] = (
                f'{outcome_name} ~ frontier_year_c + {baseline}_c '
                f'+ log_cane_area{topo_term} + {prior_formula} + C({uf_b})'
            )

        for model_name, formula in model_formulas.items():
            terms = [
                outcome_name, 'frontier_year_c',
                f'{baseline}_c', 'log_cane_area', uf_b
            ]
            topography_included = 'share_cane_slope_le12' in formula
            if topography_included:
                terms.append('share_cane_slope_le12')
            prior_included = 'uso anterior' in model_name
            if prior_included:
                terms.extend(prior_terms_b)

            terms = [c for c in terms if c in sample_b.columns]
            fit_b = sample_b.dropna(subset=terms).copy()

            if len(fit_b) < 20:
                buffer_missing.append({
                    'buffer_km': b,
                    'outcome': outcome_name,
                    'model': model_name,
                    'reason': f'insufficient_complete_cases_n_{len(fit_b)}'
                })
                continue

            model_b = smf.ols(formula, data=fit_b).fit(cov_type='HC3')
            ci = model_b.conf_int()

            if 'frontier_year_c' in model_b.params.index:
                buffer_robust_rows.append({
                    'buffer_km': b,
                    'outcome': outcome_name,
                    'model': model_name,
                    'n': int(model_b.nobs),
                    'coef_frontier_year': model_b.params['frontier_year_c'],
                    'std_err_frontier_year': model_b.bse['frontier_year_c'],
                    'p_value_frontier_year': model_b.pvalues['frontier_year_c'],
                    'ci_low_frontier_year': ci.loc['frontier_year_c', 0],
                    'ci_high_frontier_year': ci.loc['frontier_year_c', 1],
                    'effect_per_decade': model_b.params['frontier_year_c'] * 10,
                    'ci_low_per_decade': ci.loc['frontier_year_c', 0] * 10,
                    'ci_high_per_decade': ci.loc['frontier_year_c', 1] * 10,
                    'r2': model_b.rsquared,
                    'r2_adj': model_b.rsquared_adj,
                    'aic': model_b.aic,
                    'bic': model_b.bic,
                    'frontier_source_method': frontier_source,
                    'topography_source_method': topo_source,
                    'topography_included': topography_included,
                    'prior_use_included': prior_included,
                })

buffer_robustness_table = pd.DataFrame(buffer_robust_rows)
missing_buffer_caches = pd.DataFrame(buffer_missing)

if not buffer_robustness_table.empty:
    buffer_robustness_table.to_csv(
        FINAL_ROBUSTNESS / 'robustez_buffers_modelos_fronteira.csv',
        index=False,
        encoding='utf-8-sig'
    )

    # Versões bilíngues de apoio.
    buffer_robustness_table.to_csv(
        FINAL_TABLES_PTBR / 'Tabela_S2_robustez_buffers_modelos_ptbr.csv',
        index=False,
        encoding='utf-8-sig'
    )
    buffer_robustness_table.to_csv(
        FINAL_TABLES_EN / 'Table_S2_buffer_robustness_models_en.csv',
        index=False,
        encoding='utf-8-sig'
    )

    display(buffer_robustness_table.round(4))

    # Figura principal: robustez do efeito da fronteira sobre ΔNEEA.
    # Para manter comparabilidade, a figura prioriza M3 e M5b quando buffers ampliados
    # não possuem topografia exata.
    plot_df = buffer_robustness_table.loc[
        buffer_robustness_table['outcome'].eq('delta_neea')
    ].copy()

    if not plot_df.empty:
        preferred_order = [
            'M3 — + escala e UF',
            'M4 — + topografia',
            'M5 — + uso anterior',
            'M5b — + uso anterior sem topografia'
        ]
        plot_df['model_order'] = plot_df['model'].map({m: i for i, m in enumerate(preferred_order)}).fillna(99)

        for lang in LANGS:
            fig, ax = plt.subplots(figsize=(8.8, 5.4))
            for model_name, g in plot_df.sort_values(['model_order', 'buffer_km']).groupby('model', sort=False):
                g = g.sort_values('buffer_km')
                yerr = np.vstack([
                    g['effect_per_decade'] - g['ci_low_per_decade'],
                    g['ci_high_per_decade'] - g['effect_per_decade']
                ])
                ax.errorbar(
                    g['buffer_km'],
                    g['effect_per_decade'],
                    yerr=yerr,
                    marker='o',
                    linewidth=1.5,
                    capsize=3,
                    label=model_name
                )

            ax.axhline(0, linestyle='--', linewidth=1)

            if lang == 'ptbr':
                ax.set_xlabel('Buffer ao redor da unidade certificada (km)')
                ax.set_ylabel('Efeito da fronteira por década sobre ΔNEEA')
                ax.set_title('Robustez multiescala do efeito da fronteira histórica')
                ax.text(
                    0.01, -0.22,
                    'Nota: buffers sem DEM exato são estimados sem a covariável topográfica.',
                    transform=ax.transAxes,
                    ha='left', va='top', fontsize=8
                )
                stem = 'Figura_S2_robustez_buffer_efeito_fronteira_NEEA'
            else:
                ax.set_xlabel('Buffer around certified unit (km)')
                ax.set_ylabel('Frontier effect per decade on ΔNEEA')
                ax.set_title('Multiscale robustness of the historical-frontier effect')
                ax.text(
                    0.01, -0.22,
                    'Note: buffers without exact DEM cache are estimated without the topographic covariate.',
                    transform=ax.transAxes,
                    ha='left', va='top', fontsize=8
                )
                stem = 'Figure_S2_buffer_robustness_frontier_effect_NEEA'

            ax.legend(frameon=False)
            add_panel_grid(ax)
            fig.tight_layout()
            save_lang_figure(fig, stem, lang)
            plt.show()

if not missing_buffer_caches.empty:
    missing_buffer_caches.to_csv(
        FINAL_ROBUSTNESS / 'missing_buffer_caches.csv',
        index=False,
        encoding='utf-8-sig'
    )
    print('Observações sobre buffers/caches/modelos:')
    display(missing_buffer_caches)


In [ ]:
# ============================================================
# 6.3. FRONTEIRA CONTÍNUA, ΔNEEA E MÉTRICA CONTRASTIVA ΔVOL%
# ============================================================
def _get_frontier_model_for(outcome_name):
    preferred = [
        (outcome_name, 'M5 — + uso anterior'),
        (outcome_name, 'M4 — + topografia'),
        (outcome_name, 'M3 — + escala e UF'),
        (outcome_name, 'M2 — + nível inicial'),
        (outcome_name, 'M1 — fronteira'),
    ]
    for key in preferred:
        if key in model_objects:
            return key, model_objects[key]
    return None, None


def plot_frontier_continuous(outcome_name, lang='ptbr', is_main_figure=True):
    key, model_pair = _get_frontier_model_for(outcome_name)
    sample = main.loc[main['outcome'].astype(str).eq(outcome_name)].copy()
    sample = sample.dropna(subset=['mean_first_key_year', outcome_name])

    if sample.empty:
        print(f'Sem amostra para {outcome_name}.')
        return

    fig, ax = plt.subplots(figsize=(8.2, 5.4))
    ax.scatter(
        sample['mean_first_key_year'],
        sample[outcome_name],
        s=22,
        alpha=.55,
        edgecolor='none'
    )

    x_line = np.linspace(sample['mean_first_key_year'].min(), sample['mean_first_key_year'].max(), 100)

    annotation = None
    annotation_p = np.nan
    annotation_is_adjusted = False
    if model_pair is not None:
        model, fit_sample = model_pair
        beta = model.params.get('frontier_year_c', np.nan)
        annotation_p = model.pvalues.get('frontier_year_c', np.nan)
        if np.isfinite(beta):
            x_mean = fit_sample['mean_first_key_year'].mean()
            y_mean = fit_sample[outcome_name].mean()
            y_line = y_mean + beta * (x_line - x_mean)
            ax.plot(x_line, y_line, linewidth=2.0)
            annotation = beta * 10
            annotation_is_adjusted = True

    # Linha descritiva simples, apenas como referência visual se o modelo principal não existir.
    if annotation is None and len(sample) > 5:
        simple = smf.ols(f'{outcome_name} ~ mean_first_key_year', data=sample).fit()
        ax.plot(x_line, simple.predict(pd.DataFrame({'mean_first_key_year': x_line})), linewidth=2.0)
        annotation = simple.params.get('mean_first_key_year', np.nan) * 10
        annotation_p = simple.pvalues.get('mean_first_key_year', np.nan)
        annotation_is_adjusted = False

    if lang == 'ptbr':
        ax.set_xlabel('Ano médio da fronteira canavieira no entorno da unidade')
        ylab = 'ΔNEEA' if outcome_name == 'delta_neea' else 'Δvol%'
        ax.set_ylabel(ylab)
        if outcome_name == 'delta_neea':
            ax.set_title('Relação entre fronteira canavieira e ΔNEEA')
            stem = 'Figura_6_fronteira_continua_delta_NEEA'
        else:
            ax.set_title('Relação entre fronteira canavieira e Δvol%')
            stem = 'Figura_S3_fronteira_continua_delta_vol'
        if annotation is not None and np.isfinite(annotation):
            if annotation_is_adjusted and np.isfinite(annotation_p) and annotation_p >= 0.05:
                note = f'Inclinação ajustada ≈ {annotation:+.2f} por década (n.s.)'
            elif annotation_is_adjusted:
                note = f'Efeito ajustado ≈ {annotation:+.2f} por década'
            else:
                note = f'Tendência descritiva ≈ {annotation:+.2f} por década'
            ax.text(.02, .98, note, transform=ax.transAxes, va='top', fontsize=9)
    else:
        ax.set_xlabel('Mean year of sugarcane frontier around the mill')
        ylab = 'ΔNEEA' if outcome_name == 'delta_neea' else 'Δvol%'
        ax.set_ylabel(ylab)
        if outcome_name == 'delta_neea':
            ax.set_title('Relationship between sugarcane frontier and ΔNEEA')
            stem = 'Figure_6_continuous_frontier_delta_NEEA'
        else:
            ax.set_title('Relationship between sugarcane frontier and Δvol%')
            stem = 'Figure_S3_continuous_frontier_delta_vol'
        if annotation is not None and np.isfinite(annotation):
            if annotation_is_adjusted and np.isfinite(annotation_p) and annotation_p >= 0.05:
                note = f'Adjusted slope ≈ {annotation:+.2f} per decade (n.s.)'
            elif annotation_is_adjusted:
                note = f'Adjusted effect ≈ {annotation:+.2f} per decade'
            else:
                note = f'Descriptive trend ≈ {annotation:+.2f} per decade'
            ax.text(.02, .98, note, transform=ax.transAxes, va='top', fontsize=9)

    add_panel_grid(ax)
    fig.tight_layout()
    save_lang_figure(fig, stem, lang)
    plt.show()


for lang in LANGS:
    # Figura principal: ΔNEEA contínuo.
    plot_frontier_continuous('delta_neea', lang=lang, is_main_figure=True)
    # Métrica contrastiva/suplementar: Δvol%.
    plot_frontier_continuous('delta_vol_pct', lang=lang, is_main_figure=False)

# Boxplots por geração institucional: diagnóstico suplementar, não eixo principal.
generation_order = [
    'frontier_until_2001',
    'frontier_2002_2008',
    'frontier_2009_2015',
]

for outcome_name, spec in MODEL_SPECS.items():
    sample = main.loc[main['outcome'].astype(str).eq(outcome_name)].copy()

    for lang in LANGS:
        labels = get_labels(lang)
        generation_labels = [labels['frontier_period'][g] for g in generation_order]

        data = [
            sample.loc[
                sample['frontier_period_generation'].astype(str).eq(category),
                outcome_name
            ].dropna()
            for category in generation_order
        ]

        fig, ax = plt.subplots(figsize=(9, 6))
        ax.boxplot(data, labels=generation_labels, showfliers=False)

        if lang == 'ptbr':
            ax.set_xlabel('Geração institucional da fronteira')
            ax.set_ylabel(spec['label'])
            ax.set_title(f'Distribuição de {spec["label"]} por geração institucional da fronteira')
            stem = (
                'Figura_S4_delta_NEEA_por_geracao_institucional'
                if outcome_name == 'delta_neea'
                else 'Figura_S5_delta_vol_por_geracao_institucional'
            )
        else:
            ax.set_xlabel('Institutional frontier generation')
            ax.set_ylabel(spec['label'])
            ax.set_title(f'Distribution of {spec["label"]} by institutional frontier generation')
            stem = (
                'Figure_S4_delta_NEEA_by_institutional_frontier_generation'
                if outcome_name == 'delta_neea'
                else 'Figure_S5_delta_vol_by_institutional_frontier_generation'
            )

        add_panel_grid(ax)
        fig.tight_layout()
        save_lang_figure(fig, stem, lang)
        plt.show()

generation_desc = (
    main
    .groupby(['outcome', 'frontier_period_generation'], observed=True)
    .agg(
        n=('cnpj_clean', 'size'),
        frontier_mean=('mean_first_key_year', 'mean'),
        delta_neea_mean=('delta_neea', 'mean'),
        delta_neea_median=('delta_neea', 'median'),
        delta_vol_mean=('delta_vol_pct', 'mean'),
        delta_vol_median=('delta_vol_pct', 'median'),
    )
    .reset_index()
)

generation_desc['frontier_generation_label_ptbr'] = (
    generation_desc['frontier_period_generation'].astype(str).map(FRONTIER_PERIOD_LABELS_PTBR)
)
generation_desc['frontier_generation_label_en'] = (
    generation_desc['frontier_period_generation'].astype(str).map(FRONTIER_PERIOD_LABELS_EN)
)

generation_desc.to_csv(
    FINAL_TABLES / 'Tabela_9_descritivas_por_geracao_institucional.csv',
    index=False,
    encoding='utf-8-sig'
)

# Diagnóstico legado: não usar como eixo principal do artigo.
generation_desc_legacy = (
    main
    .groupby(['outcome', 'frontier_generation_legacy'], observed=True)
    .agg(
        n=('cnpj_clean', 'size'),
        frontier_mean=('mean_first_key_year', 'mean'),
        delta_neea_mean=('delta_neea', 'mean'),
        delta_vol_mean=('delta_vol_pct', 'mean'),
    )
    .reset_index()
)

generation_desc_legacy.to_csv(
    FINAL_DIAGNOSTICS / 'descritivas_por_geracao_legacy_1995_2005.csv',
    index=False,
    encoding='utf-8-sig'
)

display(generation_desc)


In [ ]:

# ============================================================
# 6.4. NÃO LINEARIDADE — DIAGNÓSTICO PARA ΔNEEA
# ============================================================
neea_sample = main.loc[
    main['outcome'].astype(str).eq('delta_neea')
].copy()

neea_sample['neea_first_c'] = (
    neea_sample['neea_first']
    - neea_sample['neea_first'].mean()
)

nonlinear_needed = [
    'delta_neea', 'mean_first_key_year', 'neea_first_c',
    'log_cane_area', 'share_cane_slope_le12', uf_col
]

nonlin = neea_sample.dropna(
    subset=nonlinear_needed
).copy()

f_linear = (
    f'delta_neea ~ mean_first_key_year + neea_first_c '
    f'+ log_cane_area + share_cane_slope_le12 + C({uf_col})'
)

f_categories = (
    f'delta_neea ~ C(frontier_period_generation) + neea_first_c '
    f'+ log_cane_area + share_cane_slope_le12 + C({uf_col})'
)

f_spline = (
    f'delta_neea ~ bs(mean_first_key_year, df=4, degree=3, '
    f'include_intercept=False) + neea_first_c + log_cane_area '
    f'+ share_cane_slope_le12 + C({uf_col})'
)

m_linear_nr = smf.ols(f_linear, data=nonlin).fit()
m_categories_nr = smf.ols(f_categories, data=nonlin).fit()
m_spline_nr = smf.ols(f_spline, data=nonlin).fit()

nonlinear_comparison = pd.DataFrame([
    {
        'model': 'Linear',
        'AIC': m_linear_nr.aic,
        'BIC': m_linear_nr.bic,
        'R2': m_linear_nr.rsquared,
        'R2_adj': m_linear_nr.rsquared_adj,
        'n': int(m_linear_nr.nobs)
    },
    {
        'model': 'Categorias institucionais',
        'AIC': m_categories_nr.aic,
        'BIC': m_categories_nr.bic,
        'R2': m_categories_nr.rsquared,
        'R2_adj': m_categories_nr.rsquared_adj,
        'n': int(m_categories_nr.nobs)
    },
    {
        'model': 'Spline cúbica',
        'AIC': m_spline_nr.aic,
        'BIC': m_spline_nr.bic,
        'R2': m_spline_nr.rsquared,
        'R2_adj': m_spline_nr.rsquared_adj,
        'n': int(m_spline_nr.nobs)
    },
])

nonlinear_comparison['delta_AIC'] = (
    nonlinear_comparison['AIC']
    - nonlinear_comparison['AIC'].min()
)
nonlinear_comparison['delta_BIC'] = (
    nonlinear_comparison['BIC']
    - nonlinear_comparison['BIC'].min()
)

nonlinear_comparison.to_csv(
    FINAL_DIAGNOSTICS / 'comparacao_linear_categorias_institucionais_spline_NEEA.csv',
    index=False,
    encoding='utf-8-sig'
)

linear_vs_spline = anova_lm(m_linear_nr, m_spline_nr)
linear_vs_spline.to_csv(
    FINAL_DIAGNOSTICS / 'anova_linear_vs_spline_NEEA.csv'
)

display(nonlinear_comparison.round(4))
display(linear_vs_spline)


In [ ]:

# ============================================================
# 6.5. MORAN GLOBAL E LISA DOS RESÍDUOS — DIAGNÓSTICO KNN
# ============================================================
candidate_keys = [
    ('delta_neea', 'M5 — + uso anterior'),
    ('delta_neea', 'M4 — + topografia'),
    ('delta_neea', 'M3 — + escala e UF'),
]

final_key = next(
    (key for key in candidate_keys if key in model_objects),
    None
)

if final_key is None:
    print('Nenhum modelo de ΔNEEA disponível para diagnóstico espacial.')
    moran_table_all = pd.DataFrame()
    lisa_counts_all = pd.DataFrame()
else:
    neea_model, neea_fit = model_objects[final_key]
    neea_fit = neea_fit.copy()

    neea_fit['residual'] = neea_model.resid
    neea_fit['fitted'] = neea_model.fittedvalues

    # Usar coordenadas métricas para KNN. Se x/y estiverem em lon/lat, projeta para EPSG:5880;
    # se já estiverem projetadas, a rotina infere o CRS e também projeta para EPSG:5880.
    try:
        neea_points = build_points_geodataframe(
            neea_fit,
            id_cols=['cnpj_clean'],
            label='resíduos ΔNEEA'
        ).to_crs('EPSG:5880')
        coords = np.column_stack([neea_points.geometry.x, neea_points.geometry.y])
        print(
            'Coordenadas do diagnóstico Moran/LISA projetadas em EPSG:5880; '
            f'CRS original inferido: {neea_points["coord_crs_used"].iloc[0]}'
        )
    except Exception as exc:
        print('AVISO: falha ao projetar coordenadas para EPSG:5880; usando x/y brutos.', exc)
        coords = neea_fit[[x_col, y_col]].to_numpy()

    moran_rows = []
    lisa_count_rows = []
    lisa_outputs = {}

    for k in CFG['k_neighbors_robustness']:
        k_eff = min(k, len(neea_fit) - 1)

        weights = KNN.from_array(coords, k=k_eff)
        weights.transform = 'R'

        seed_spatial_inference(k_eff * 100)
        moran = Moran(
            neea_fit['residual'].to_numpy(),
            weights,
            permutations=CFG['permutations']
        )

        moran_rows.append({
            'model': final_key[1],
            'n': len(neea_fit),
            'k_neighbors': k_eff,
            'moran_i': moran.I,
            'expected_i': moran.EI,
            'p_permutation': moran.p_sim,
            'z_sim': moran.z_sim,
            'permutations': CFG['permutations'],
            'is_main_knn': k_eff == CFG['k_neighbors'],
        })

        seed_spatial_inference(k_eff * 100 + 1)
        local = Moran_Local(
            neea_fit['residual'].to_numpy(),
            weights,
            permutations=CFG['permutations']
        )

        reject, p_adj, _, _ = multipletests(
            local.p_sim,
            alpha=CFG['fdr_alpha'],
            method='fdr_bh'
        )

        quadrants = {1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}

        tmp = neea_fit.copy()
        tmp['k_neighbors'] = k_eff
        tmp['local_i'] = local.Is
        tmp['local_p_raw'] = local.p_sim
        tmp['local_p_fdr'] = p_adj
        tmp['lisa_residual_class'] = [
            quadrants.get(q, 'NS') if significant else 'NS'
            for q, significant in zip(local.q, reject)
        ]

        counts = (
            tmp['lisa_residual_class']
            .value_counts()
            .rename_axis('lisa_residual_class')
            .reset_index(name='count')
        )
        counts['k_neighbors'] = k_eff
        counts['model'] = final_key[1]
        counts['is_main_knn'] = k_eff == CFG['k_neighbors']
        lisa_count_rows.append(counts)

        lisa_outputs[k_eff] = tmp

    moran_table_all = pd.DataFrame(moran_rows)
    lisa_counts_all = pd.concat(lisa_count_rows, ignore_index=True)

    moran_table_all.to_csv(
        FINAL_DIAGNOSTICS / 'moran_global_residuos_NEEA_knn_robustness.csv',
        index=False,
        encoding='utf-8-sig'
    )

    lisa_counts_all.to_csv(
        FINAL_DIAGNOSTICS / 'lisa_residuos_NEEA_counts_knn_robustness.csv',
        index=False,
        encoding='utf-8-sig'
    )

    main_k = min(CFG['k_neighbors'], len(neea_fit) - 1)
    neea_fit_lisa_main = lisa_outputs[main_k]

    neea_fit_lisa_main.to_csv(
        FINAL_DIAGNOSTICS / 'residuos_NEEA_com_LISA_knn6.csv',
        index=False,
        encoding='utf-8-sig'
    )

    # Arquivos com nomes legados para compatibilidade.
    moran_table = moran_table_all.loc[
        moran_table_all['k_neighbors'].eq(main_k)
    ].copy()
    moran_table.to_csv(
        FINAL_DIAGNOSTICS / 'moran_global_residuos_NEEA.csv',
        index=False,
        encoding='utf-8-sig'
    )
    neea_fit_lisa_main.to_csv(
        FINAL_DIAGNOSTICS / 'residuos_NEEA_com_LISA.csv',
        index=False,
        encoding='utf-8-sig'
    )

    display(moran_table_all.round(4))
    display(lisa_counts_all)



In [ ]:

# ============================================================
# 6.5. ROBUSTEZ DE ΔVOL% A OBSERVAÇÕES INFLUENTES
# ============================================================
vol_sample = main.loc[
    main['outcome'].astype(str).eq('delta_vol_pct')
].copy()

vol_sample['vol_pct_first_c'] = (
    vol_sample['vol_pct_first']
    - vol_sample['vol_pct_first'].mean()
)

vol_formula = (
    f'delta_vol_pct ~ frontier_year_c + vol_pct_first_c '
    f'+ log_cane_area + share_cane_slope_le12 + C({uf_col})'
)

vol_complete = vol_sample.dropna(
    subset=[
        'delta_vol_pct', 'frontier_year_c',
        'vol_pct_first_c', 'log_cane_area',
        'share_cane_slope_le12', uf_col
    ]
).copy()

plain = smf.ols(vol_formula, data=vol_complete).fit()
influence = OLSInfluence(plain)

vol_complete['student_resid'] = (
    influence.resid_studentized_external
)
vol_complete['cooks_d'] = influence.cooks_distance[0]
vol_complete['hat_diag'] = influence.hat_matrix_diag

n = len(vol_complete)
p = int(plain.df_model) + 1

vol_complete['outlier_any'] = (
    vol_complete['student_resid'].abs().gt(3)
    | vol_complete['cooks_d'].gt(4 / n)
    | vol_complete['hat_diag'].gt(2 * p / n)
)

vol_no_influential = vol_complete.loc[
    ~vol_complete['outlier_any']
].copy()

q01, q99 = vol_complete['delta_vol_pct'].quantile([.01, .99])

vol_trim = vol_complete.loc[
    vol_complete['delta_vol_pct'].between(q01, q99)
].copy()

vol_winsor = vol_complete.copy()
vol_winsor['delta_vol_pct_w'] = (
    vol_winsor['delta_vol_pct'].clip(q01, q99)
)

robust_models = {
    'Original HC3': smf.ols(
        vol_formula, data=vol_complete
    ).fit(cov_type='HC3'),
    'Sem influentes HC3': smf.ols(
        vol_formula, data=vol_no_influential
    ).fit(cov_type='HC3'),
    'Trim 1–99 HC3': smf.ols(
        vol_formula, data=vol_trim
    ).fit(cov_type='HC3'),
    'Winsor HC3': smf.ols(
        vol_formula.replace(
            'delta_vol_pct ~',
            'delta_vol_pct_w ~'
        ),
        data=vol_winsor
    ).fit(cov_type='HC3'),
    'RLM Huber': smf.rlm(
        vol_formula,
        data=vol_complete,
        M=HuberT()
    ).fit(),
}

robust_rows = []

for name, model in robust_models.items():
    ci = model.conf_int()
    for term in model.params.index:
        robust_rows.append({
            'model': name,
            'term': term,
            'coef': model.params[term],
            'std_err': model.bse[term],
            'p_value': model.pvalues[term],
            'ci_low': ci.loc[term, 0],
            'ci_high': ci.loc[term, 1],
            'n': int(model.nobs),
            'r2': getattr(model, 'rsquared', np.nan),
        })

vol_robustness = pd.DataFrame(robust_rows)

vol_robustness.to_csv(
    FINAL_DIAGNOSTICS / 'robustez_delta_vol_outliers.csv',
    index=False,
    encoding='utf-8-sig'
)

vol_complete.to_csv(
    FINAL_DIAGNOSTICS / 'diagnostico_influencia_delta_vol.csv',
    index=False,
    encoding='utf-8-sig'
)

display(
    vol_robustness.loc[
        vol_robustness['term'].eq('frontier_year_c')
    ].round(4)
)



## 7. Tabelas consolidadas e pacote final


In [ ]:
# ============================================================
# 7.1. WORKBOOK DE TABELAS PRINT-READY
# ============================================================
workbook_path = FINAL_DIR / 'Tabelas_artigo_AmbSoc_print_ready.xlsx'

PERIOD_RANK = {p: i for i, p in enumerate(PERIOD_ORDER)}
SLOPE_RANK = {s: i for i, s in enumerate(SLOPE_ORDER)}

def sort_for_workbook(df):
    df = df.copy()
    if 'period_order' in df.columns:
        sort_cols = ['period_order']
        if 'origin_order' in df.columns:
            sort_cols.append('origin_order')
        elif 'slope_bin' in df.columns:
            df['_slope_order'] = df['slope_bin'].map(SLOPE_RANK)
            sort_cols.append('_slope_order')
        return df.sort_values(sort_cols).drop(columns=['_slope_order'], errors='ignore')
    if 'period' in df.columns:
        df['_period_order'] = df['period'].map(PERIOD_RANK)
        sort_cols = ['_period_order']
        if 'origin_code' in df.columns:
            df['_origin_order'] = df['origin_code'].replace({99: 999})
            sort_cols.append('_origin_order')
        if 'slope_bin' in df.columns:
            df['_slope_order'] = df['slope_bin'].map(SLOPE_RANK)
            sort_cols.append('_slope_order')
        df = df.sort_values(sort_cols)
        return df.drop(columns=['_period_order', '_origin_order', '_slope_order'], errors='ignore')
    if 'buffer_km' in df.columns:
        sort_cols = ['buffer_km']
        if 'outcome' in df.columns:
            sort_cols.append('outcome')
        if 'model' in df.columns:
            sort_cols.append('model')
        return df.sort_values(sort_cols)
    return df

notes = pd.DataFrame([
    {
        'item': 'Temporalidade principal',
        'note_ptbr': 'As análises de expansão, topografia e uso anterior usam períodos institucionais. Anos-chave são snapshots cartográficos/contextuais.',
        'note_en': 'Expansion, topography, and previous land-use analyses use institutional periods. Key years are cartographic/contextual snapshots.',
    },
    {
        'item': 'Uso anterior — denominador',
        'note_ptbr': 'origin_share usa o total de entrada persistente da T2. A classe 99 fecha a pequena diferença entre esse total e a soma das sete classes mapeadas.',
        'note_en': 'origin_share uses the total persistent-entry area from T2. Class 99 closes the small gap between this total and the seven mapped origin classes.',
    },
    {
        'item': 'Buffer principal',
        'note_ptbr': f'O buffer ativo desta execução foi {CFG["active_buffer_km"]} km. Quando 30 km não está disponível, o notebook usa o cache legado de 25 km como fallback.',
        'note_en': f'The active buffer in this run was {CFG["active_buffer_km"]} km. When 30 km is unavailable, the notebook uses the legacy 25 km cache as fallback.',
    },
    {
        'item': 'Robustez multibuffer',
        'note_ptbr': 'Buffers 30/50/100/150 km podem usar aproximação por grade de 50 km quando caches exatos não existem. A tabela indica o método de origem.',
        'note_en': 'Buffers 30/50/100/150 km may use a 50-km grid approximation when exact caches are unavailable. The table reports the source method.',
    },
    {
        'item': 'Moran/LISA',
        'note_ptbr': 'O diagnóstico espacial principal usa KNN=6, com robustez para KNN=4, 8 e 10.',
        'note_en': 'The main spatial diagnostic uses KNN=6, with robustness checks for KNN=4, 8, and 10.',
    },
])

tables_to_write = {
    'README_Notas': notes,
    'T1_Descritivas': descriptives,
    'T2_Expansao': expansion_period,
    'T3_Selecao_topo': period_topo,
    'T4_Contrastes_topo': topo_contrasts,
    'T7_Modelos': model_table,
    'T8_Fronteira_decada': frontier_terms,
    'T9_Geracao_institucional': generation_desc,
}

optional_tables = {
    'T5_Uso_anterior': globals().get('origin_summary'),
    'T6_Contrastes_uso': globals().get('origin_contrasts'),
    'T10_Robustez_buffers': globals().get('buffer_robustness_table'),
    'T11_Moran_KNN': globals().get('moran_table_all'),
    'T12_LISA_residuos_KNN': globals().get('lisa_counts_all'),
    'T13_Uso_ant_unidade': globals().get('unit_origin_all_buffers'),
    'T14_Fronteira_aprox': globals().get('unit_frontier_approx_all_buffers'),
    'T15_Amostras_modelos': globals().get('model_sample_table'),
}

for name, obj in optional_tables.items():
    if isinstance(obj, pd.DataFrame) and not obj.empty:
        tables_to_write[name] = obj

with pd.ExcelWriter(workbook_path, engine='openpyxl') as writer:
    for sheet, df in tables_to_write.items():
        # Excel limita sheet_name a 31 caracteres.
        sort_for_workbook(df).to_excel(writer, sheet_name=sheet[:31], index=False)

# Formatação básica para revisão e submissão.
wb = load_workbook(workbook_path)

header_fill = PatternFill(start_color='D9EAF7', end_color='D9EAF7', fill_type='solid')
header_font = Font(bold=True)

for ws in wb.worksheets:
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

    for col in ws.columns:
        max_len = 0
        column = col[0].column_letter
        for cell in col:
            try:
                value_len = len(str(cell.value)) if cell.value is not None else 0
                max_len = max(max_len, value_len)
            except Exception:
                pass
        ws.column_dimensions[column].width = min(max(max_len + 2, 10), 48)

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            if isinstance(cell.value, float):
                cell.number_format = '0.0000'
            cell.alignment = Alignment(vertical='top', wrap_text=False)

wb.save(workbook_path)

print('Workbook print-ready:', workbook_path)


In [ ]:
# ============================================================
# 7.2. MANIFESTO E ZIP FINAL
#
# O pacote consolidado é gravado FORA de outputs/. Escrever o zip dentro do
# diretório que ele comprime faz o `rglob` encontrá-lo durante a escrita, e o
# arquivo cresce indefinidamente.
# ============================================================

DIST_DIR = ROOT / 'dist'
DIST_DIR.mkdir(parents=True, exist_ok=True)

zip_path = DIST_DIR / 'AmbSoc_artigo_consolidado_local_multiescala.zip'


def _empacotavel(path):
    """Arquivo comum, fora de qualquer arquivo compactado."""
    return path.is_file() and path.suffix.lower() not in {'.zip', '.7z', '.gz'}


# ---- manifesto -------------------------------------------------------------
manifest_rows = [
    {
        'filename': p.name,
        'relative_path': str(p.relative_to(FINAL_DIR)),
        'extension': p.suffix.lower(),
        'size_bytes': p.stat().st_size,
    }
    for p in sorted(FINAL_DIR.rglob('*')) if _empacotavel(p)
]

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(
    FINAL_DIR / 'output_manifest.csv',
    index=False,
    encoding='utf-8-sig'
)

# ---- zip -------------------------------------------------------------------
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(FINAL_DIR.rglob('*')):
        if _empacotavel(p):
            archive.write(p, arcname=str(p.relative_to(FINAL_DIR)))

_mb = zip_path.stat().st_size / 1e6
print(f'Arquivos finais: {len(manifest)}')
print(f'ZIP: {zip_path}  ({_mb:.1f} MB)')

if _mb > 500:
    raise RuntimeError(
        f'ZIP com {_mb:.0f} MB — muito acima do esperado (~100 MB). '
        'Verifique se algum arquivo compactado foi incluído no empacotamento.'
    )

display(manifest)


## 8. Leitura rápida esperada

A v4 foi organizada para reduzir ambiguidade temporal e deixar os outputs prontos para submissão.

Resultados esperados:

1. A expansão, a seleção topográfica e o uso anterior são apresentados por períodos institucionais.
2. A figura principal de uso anterior é o heatmap; as barras empilhadas ficam como suplemento.
3. A figura principal de desempenho mostra a relação contínua entre ano médio da fronteira e ΔNEEA.
4. Δvol% é mantido como métrica contrastiva/suplementar no AmbSoc.
5. A T5 fecha o denominador da T2 com classe residual 99 quando necessário.
6. A T1 descreve amostras analíticas por outcome, não registros duplicados da base longa.
7. Moran/LISA usa KNN=6 como principal, com robustez KNN=4/8/10.
8. O workbook final contém uma aba README_Notas com as principais cautelas metodológicas.
